# Hotel-Analytics mit pandas

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StefanHanisch/datenorganisation_und_datenbanken_2026/blob/main/notebooks/hotel_analytics.ipynb)

Unsere Hoteldatenbank ist voll: Städte, Hotels, Zimmerkategorien, Gäste, Reservierungen, Tagespreise und ein Kalender mit Feiertagen und Events. In diesem Notebook schaut ihr mit **pandas** hinein. Wenige Zeilen Code, viele Plots, keine Statistikformeln.

**Wichtig:** Die Termine der Events und Feiertage sind echt. Ihre Effekte auf Preise und Buchungen sind dagegen synthetisch erzeugt. Alle Personen, Firmen und Buchungen sind frei erfunden.

| Block | Thema | Zeit |
|---|---|---|
| 1 | Wer sind unsere Gäste? | ca. 1 h 20 Min. |
| 1b | SQL und pandas – direkt in der Datenbank | ca. 40 Min. |
| 2 | Kalender-Detektive | ca. 1 h 25 Min. |
| 3 | Preisfalle und Umsatzkurve | ca. 1 h 55 Min. |
| 4 | SpiegelMining im Hotel (Vertiefung) | ca. 1 h 45 Min. |
| 5 | Darf man das? | ca. 15 Min. |
| | **Gesamt** | **ca. 7 h 20 Min.** |

Die Zeiten sind großzügig geschätzt. Lasst euch Zeit, lest in Ruhe und probiert aus. Block 4 ist eine Vertiefung für alle, die noch Lust haben.

## So funktioniert dieses Notebook

- Führt die Zellen **von oben nach unten** aus: Zelle anklicken und **Shift+Enter** drücken.
- Vor jeder Aufgabe steht ein **Mini-Beispiel** mit einem winzigen DataFrame. Probiert es aus und verändert es ruhig. So seht ihr, was eine Funktion macht, bevor ihr sie auf die echten Daten loslasst.
- In den Aufgaben-Zellen ersetzt ihr jedes `___` (drei Unterstriche) durch euren Code. Der Kommentar `# TODO:` sagt euch, was gesucht ist.
- Danach folgt eine **Check-Zelle**. Sie meldet ✅, wenn alles passt, oder gibt euch mit 💡 einen Tipp.
- `LueckeOffen: In dieser Zelle steht noch eine Lücke ___` heißt: Hier fehlt noch eure Lösung. Bei Lücken an manchen Stellen meldet pandas stattdessen einen eigenen Fehler, in dem `___` oder `OpenGap` vorkommt. Auch das heißt: Lücke noch offen.
- `NameError: name '…' is not defined` in einer Check- oder Plot-Zelle heißt meist: Die Aufgabe darüber ist noch nicht gelöst.
- Wenn alles durcheinander ist: Menü **Laufzeit/Kernel → Neu starten** und dann alle Zellen von oben ausführen. Die Daten kommen dann aus dem Cache, das geht schnell.

## So startet ihr

Ihr braucht die **Datenbank-URL** von eurer Lehrperson. Sie beginnt mit `postgresql://` und enthält ein Passwort. Schreibt sie nie in das Notebook und committet sie nie.

### Weg A: Google Colab (nichts installieren)

1. Klickt oben auf den Button **In Colab öffnen**.
2. Gebt die URL auf **einem** dieser Wege an:
   - **Secrets:** Links auf das Schlüsselsymbol klicken, ein Secret mit dem Namen `AZURE_ANALYST_DATABASE_URL` anlegen, die URL als Wert eintragen und den Notebook-Zugriff erlauben.
   - **.env-Datei:** Links auf das Ordnersymbol klicken und eine Datei `.env` hochladen, die die Zeile `AZURE_ANALYST_DATABASE_URL=postgresql://…` enthält.
   - **Eingabe:** Einfach die Setup-Zelle ausführen. Wenn nichts gefunden wird, fragt sie nach der URL. Die Eingabe bleibt verdeckt.
3. Führt die Setup-Zelle unten aus.

### Weg B: lokal in VS Code

1. Python 3.11 oder neuer installieren.
2. Im Projektordner einmal ausführen: `pip install -e ".[analytics]"`
3. Im Projektordner eine Datei `.env` mit der Zeile `AZURE_ANALYST_DATABASE_URL=postgresql://…` anlegen. Die Datei ist von Git ausgeschlossen.
4. Das Notebook öffnen, oben rechts den Python-Kernel wählen und die Setup-Zelle ausführen.

## Spickzettel

| Befehl | wofür | Beispiel |
|---|---|---|
| `df.head()` | die ersten Zeilen ansehen | `reservations.head(3)` |
| `df["spalte"]` | eine Spalte auswählen | `reservations["status"]` |
| `df.query("x > 3")` | Zeilen filtern | `stays.query("nights > 3")` |
| `df.merge(other, on=...)` | zwei Tabellen verbinden (Join) | `stays.merge(guests, left_on="guest_id", right_on="id")` |
| `df.groupby("a")["b"].mean()` | Gruppen bilden und zusammenfassen | `stays.groupby("city")["nights"].mean()` |
| `df.pivot_table(index=, columns=, values=, aggfunc=)` | Kreuztabelle | `stays.pivot_table(index="month", columns="year", values="id", aggfunc="count")` |
| `np.where(bed, a, b)` | Wert je nach Bedingung | `np.where(stays["nights"] > 3, "lang", "kurz")` |
| `pd.cut(werte, bins=..., labels=...)` | Zahlen in Klassen einteilen | `pd.cut(stays["nights"], bins=[0, 2, 7, 30])` |
| `.dt.year`, `.dt.day_of_week` | Teile eines Datums (0 = Montag) | `stays["checkin"].dt.year` |
| `.plot(...)`, `.plot.bar()`, `.plot.hist()` | Linie, Balken, Histogramm | `stays["nights"].plot.hist(title="Nächte")` |
| `pd.read_sql(text(...), engine)` | SQL an die Datenbank schicken | `pd.read_sql(text("SELECT * FROM analytics.cities"), engine)` |

### SQL ↔ pandas

| SQL | pandas |
|---|---|
| `SELECT name, stars FROM hotels` | `hotels[["name", "stars"]]` |
| `WHERE stars >= 4` | `hotels.query("stars >= 4")` |
| `JOIN cities ON hotels.city_id = cities.id` | `hotels.merge(cities, left_on="city_id", right_on="id")` |
| `GROUP BY city_id` mit `AVG(stars)` | `hotels.groupby("city_id")["stars"].mean()` |
| `ORDER BY stars DESC` | `hotels.sort_values("stars", ascending=False)` |

In [ ]:
# @title Setup – einfach ausführen (lädt Pakete und verbindet mit der Datenbank)
import getpass
import os
import re
import subprocess
import sys
import time
import urllib.parse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError, ProgrammingError

try:
    import psycopg2  # noqa: F401  (Datenbank-Treiber für SQLAlchemy)
except ImportError:
    print("Installiere den Datenbank-Treiber psycopg2-binary …")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "psycopg2-binary"])
    import psycopg2  # noqa: F401

plt.rcParams["figure.figsize"] = (9, 4)
pd.set_option("display.max_columns", 30)

ENV_NAME = "AZURE_ANALYST_DATABASE_URL"


def find_project_root():
    """Sucht von hier aus nach oben den Ordner mit pyproject.toml (sonst None)."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    return None


def read_env_file(path):
    """Liest KEY=VALUE-Zeilen aus einer .env-Datei in ein dict."""
    values = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export "):].strip()
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        key, value = key.strip(), value.strip()
        # Optionale Anführungszeichen entfernen: KEY="wert" oder KEY='wert'
        if len(value) >= 2 and value[0] == value[-1] and value[0] in "\"'":
            value = value[1:-1]
        values[key] = value
    return values


def env_file_candidates(project_root):
    """Mögliche .env-Dateien: aktueller Ordner, Eltern bis zum Projektordner, Colab."""
    folders = [Path.cwd()]
    if project_root is not None:
        for folder in Path.cwd().parents:
            if project_root == folder or project_root in folder.parents:
                folders.append(folder)
    folders.append(Path("/content"))  # Ordner für Uploads in Colab
    return [folder / ".env" for folder in folders]


def read_colab_secret():
    """Liest das Colab-Secret, falls wir in Colab sind und der Zugriff erlaubt ist."""
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(ENV_NAME)
    except Exception:
        print(f"💡 Colab-Secret {ENV_NAME} nicht gefunden oder Notebook-Zugriff nicht erlaubt.")
        return None


def get_db_url():
    """Findet die Datenbank-URL und gibt sie zurück, ohne sie anzuzeigen."""
    url, source = os.environ.get(ENV_NAME), "Umgebungsvariable"
    if not url:
        for env_file in env_file_candidates(find_project_root()):
            if env_file.is_file():
                url = read_env_file(env_file).get(ENV_NAME)
                if url:
                    source = f".env-Datei ({env_file})"
                    break
    if not url:
        url, source = read_colab_secret(), "Colab-Secret"
    if not url:
        url = getpass.getpass("Datenbank-URL (von der Lehrperson): ")
        source = "Eingabe"

    url = (url or "").strip()
    if not url.startswith("postgresql://"):
        raise ValueError(
            f"Die Datenbank-URL aus der Quelle '{source}' beginnt nicht mit 'postgresql://'. "
            "Prüft, ob ihr die vollständige URL von der Lehrperson kopiert habt."
        )
    if re.search(r"<[^<>]+>", url):
        raise ValueError(
            f"Die Datenbank-URL aus der Quelle '{source}' enthält noch einen Platzhalter wie <PASSWORT>. "
            "Ersetzt ihn durch den echten Wert."
        )
    try:
        parts = urllib.parse.urlsplit(url)
        target = f"{parts.hostname} / {parts.path.lstrip('/')}"
    except ValueError:
        target = "(Adresse nicht lesbar)"
    print(f"Zugangsdaten aus: {source}")
    print(f"Ziel: {target}")
    return url


def connect(url):
    """Erstellt die Engine und testet die Verbindung mit SELECT current_user."""
    engine = create_engine(
        url.replace("postgresql://", "postgresql+psycopg2://", 1),
        pool_pre_ping=True,
        connect_args={"connect_timeout": 15},
    )
    try:
        with engine.connect() as connection:
            user = connection.execute(text("SELECT current_user")).scalar()
    except OperationalError as error:
        details = str(error.orig).strip().splitlines()[0] if error.orig else ""
        if "password" in details.lower():
            reason = "Benutzername oder Passwort stimmen nicht. Prüft die URL."
        else:
            reason = (
                "Die Datenbank ist nicht erreichbar. Prüft eure Internetverbindung. "
                "Manche WLANs sperren den Port 5432, dann hilft z. B. ein Handy-Hotspot."
            )
        raise ConnectionError(f"❌ Keine Verbindung zur Datenbank. {reason}\nDetails: {details}") from None
    print(f"✅ Verbunden als {user}")
    return engine


TABLES = ["cities", "hotels", "room_categories", "guests", "reservations", "daily_rates", "calendar_events"]
DATE_COLUMNS = ["checkin", "checkout", "arrival_date", "event_date", "created_at", "cancelled_at"]
FLOAT_COLUMNS = ["base_price", "rate_per_night", "commission_pct", "offered_rate"]

PROJECT_ROOT = find_project_root()
CACHE_DIR = (PROJECT_ROOT / "data" / "cache") if PROJECT_ROOT else Path("hotel_cache")


def tidy_types(name, df):
    """Einheitliche Typen: Datum ohne Zeitzone, Preise als float, city_id als Int64."""
    for column in DATE_COLUMNS:
        if column in df.columns:
            values = pd.to_datetime(df[column], utc=True).dt.tz_localize(None)
            df[column] = values.astype("datetime64[ns]")
    for column in FLOAT_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column]).astype(float)
    if name == "calendar_events":
        df["city_id"] = df["city_id"].astype("Int64")  # <NA> = bundesweit
    return df


def load_tables(refresh=False):
    """Lädt alle Tabellen aus dem Cache oder (beim ersten Mal) aus der Datenbank."""
    tables = {}
    start = time.time()
    for name in TABLES:
        table_start = time.time()
        cache_file = CACHE_DIR / f"{name}.parquet"
        if cache_file.exists() and not refresh:
            df = pd.read_parquet(cache_file)
            source = "Cache"
        else:
            try:
                df = pd.read_sql(text(f"SELECT * FROM analytics.{name}"), engine)
            except ProgrammingError:
                raise RuntimeError(
                    f"❌ Tabelle analytics.{name} nicht gefunden oder kein Leserecht. "
                    "Ist das Schema analytics eingerichtet? Fragt eure Lehrperson."
                ) from None
            df = tidy_types(name, df)
            CACHE_DIR.mkdir(parents=True, exist_ok=True)
            df.to_parquet(cache_file, index=False)
            source = "DB"
        tables[name] = tidy_types(name, df)
        rows = f"{len(df):,}".replace(",", ".")
        print(f"  {name:<16} {rows:>10} Zeilen  {time.time() - table_start:5.1f} s  ({source})")
    print(f"Fertig in {time.time() - start:.1f} s")
    return tables


class LueckeOffen(Exception):
    """Wird ausgelöst, sobald eine Zelle eine noch offene Lücke benutzt."""


class OpenGap:
    """Platzhalter für Lücken in den Aufgaben.

    Jupyter nutzt den Namen der Lücke sonst selbst (für die drittletzte Ausgabe).
    Dann würde eine offene Lücke still mit einem falschen Wert weiterrechnen.
    Dieser Platzhalter meldet stattdessen sofort, dass noch etwas fehlt.
    """

    MESSAGE = ("In dieser Zelle steht noch eine Lücke " + "_" * 3 + ". "
               "Ersetzt sie durch eure Lösung und führt die Zelle erneut aus.")

    def _stop(self, *args, **kwargs):
        raise LueckeOffen(self.MESSAGE)

    def __getattr__(self, name):
        if name.startswith("_"):
            raise AttributeError(name)
        self._stop()

    def __repr__(self):
        return "<Lücke: hier fehlt noch eure Lösung>"


for _name in ["__eq__", "__ne__", "__lt__", "__le__", "__gt__", "__ge__", "__hash__", "__bool__",
              "__len__", "__iter__", "__contains__", "__getitem__", "__call__", "__index__", "__int__",
              "__float__", "__add__", "__radd__", "__sub__", "__rsub__", "__mul__", "__rmul__",
              "__truediv__", "__rtruediv__", "__floordiv__", "__rfloordiv__", "__neg__", "__array__"]:
    setattr(OpenGap, _name, OpenGap._stop)

globals()["_" * 3] = OpenGap()  # die Lücke aus den Aufgaben

engine = connect(get_db_url())
print(f"Cache-Ordner: {CACHE_DIR}")
print("✅ Setup fertig")

In [ ]:
# @title Prüf-Funktionen laden – einfach ausführen


# ---- 00_base.py ----
# Grundfunktionen für alle Prüf-Funktionen.
# Prüf-Funktionen werfen nie Exceptions: Sie geben genau eine ✅-Zeile
# oder konkrete 💡-Tipps aus.
import numpy as np  # noqa: F401
import pandas as pd


def ok(message):
    """Meldet, dass alles passt."""
    print(f"✅ {message}")


def hint(message):
    """Gibt einen Tipp aus."""
    print(f"💡 {message}")


def has_columns(df, names, df_name):
    """True, wenn df alle Spalten aus names hat, sonst Tipp und False."""
    if not isinstance(df, pd.DataFrame):
        hint(f"`{df_name}` ist kein DataFrame, sondern {type(df).__name__}.")
        return False
    missing = [name for name in names if name not in df.columns]
    if missing:
        hint(f"In `{df_name}` fehlen die Spalten: {', '.join(missing)}")
        return False
    return True


def check_tables(tables):
    """Prüft das Ergebnis von load_tables()."""
    expected = ["cities", "hotels", "room_categories", "guests",
                "reservations", "daily_rates", "calendar_events"]
    try:
        if not isinstance(tables, dict):
            hint("`tables` sollte ein dict sein. Nutzt `tables = load_tables()`.")
            return
        missing = [name for name in expected if name not in tables]
        if missing:
            hint(f"In `tables` fehlen: {', '.join(missing)}. Nutzt `tables = load_tables()`.")
            return
        empty = [name for name in expected
                 if not isinstance(tables[name], pd.DataFrame) or tables[name].empty]
        if empty:
            hint(f"Leer oder kein DataFrame: {', '.join(empty)}. "
                 "Versucht `tables = load_tables(refresh=True)`.")
            return
        reservations = tables["reservations"]
        if not has_columns(reservations, ["checkin"], "reservations"):
            return
        if not pd.api.types.is_datetime64_any_dtype(reservations["checkin"]):
            hint("`reservations['checkin']` ist kein Datum. "
                 "Versucht `tables = load_tables(refresh=True)`.")
            return
        rows = f"{len(reservations):,}".replace(",", ".")
        ok(f"Alle {len(expected)} Tabellen geladen, darunter {rows} Reservierungen.")
    except Exception as error:
        hint(f"Prüfung nicht möglich: {error}")


# ---- 01_gaeste.py ----
# Prüf-Funktionen für Block 1: Wer sind unsere Gäste?
# Nutzt ok, hint und has_columns aus checks/00_base.py (check_tables steht dort).


def _block1_number(value):
    """Ganze Zahl mit deutschem Tausenderpunkt, z. B. 12.345."""
    return f"{int(value):,}".replace(",", ".")


def check_counts(n_reservations, n_columns, reservations):
    """Aufgabe 1.1: Zeilen- und Spaltenzahl von reservations."""
    try:
        if not isinstance(reservations, pd.DataFrame):
            hint("`reservations` ist kein DataFrame. Führt die Zellen oben noch einmal aus.")
            return
        rows, cols = reservations.shape
        if not isinstance(n_reservations, (int, np.integer)):
            hint("`n_reservations` sollte eine ganze Zahl sein. `len(...)` liefert die Anzahl Zeilen eines DataFrames.")
            return
        if n_reservations != rows:
            if n_reservations == cols:
                hint("`n_reservations` ist die Zahl der Spalten. Gesucht ist die Zahl der Zeilen von `reservations`.")
            else:
                hint("`n_reservations` passt nicht zu `reservations`. Gebt `len(...)` die ganze Tabelle `reservations`.")
            return
        if not isinstance(n_columns, (int, np.integer)):
            hint("`n_columns` sollte eine ganze Zahl sein. `shape` ist ein Paar (Zeilen, Spalten), in eckigen Klammern wählt ihr eine Position.")
            return
        if n_columns != cols:
            if n_columns == rows:
                hint("`n_columns` ist die Zahl der Zeilen. Welche Position im Paar `shape` steht für die Spalten? Python zählt ab 0.")
            else:
                hint("`n_columns` passt nicht. Schaut euch das Mini-Beispiel zu `shape` noch einmal an.")
            return
        ok(f"{_block1_number(rows)} Reservierungen mit je {cols} Spalten.")
    except Exception as error:
        hint(f"Prüfung nicht möglich: {error}")


def check_hotel_cities(hotel_cities, hotels):
    """Aufgabe 1.2: hotels mit towns verbunden."""
    try:
        if not has_columns(hotel_cities, ["name", "city", "city_type"], "hotel_cities"):
            hint("Verbindet `hotels` mit `towns`, dann kommt die Spalte `city` dazu.")
            return
        if not isinstance(hotels, pd.DataFrame):
            hint("`hotels` ist kein DataFrame. Führt Aufgabe 1.1 noch einmal aus.")
            return
        if len(hotel_cities) != len(hotels):
            hint(f"`hotel_cities` hat {len(hotel_cities)} Zeilen, es gibt aber {len(hotels)} Hotels. "
                 "Jedes Hotel sollte genau einmal vorkommen. Stimmt die Schlüsselspalte?")
            return
        if hotel_cities["city"].isna().any():
            hint("Bei einigen Hotels fehlt die Stadt. Stimmt die Schlüsselspalte?")
            return
        ok(f"Alle {len(hotel_cities)} Hotels haben ihre Stadt.")
    except Exception as error:
        hint(f"Prüfung nicht möglich: {error}")


def check_stays(stays):
    """Aufgabe 1.3: Arbeitstabelle stays."""
    try:
        needed = ["category_id", "hotel_id", "category", "city", "city_type",
                  "nights", "persons", "lead_days", "weekday", "month", "year"]
        if not has_columns(stays, needed, "stays"):
            hint("Führt die Zellen in Aufgabe 1.2 und 1.3 noch einmal der Reihe nach aus.")
            return
        if len(stays) == 0:
            hint("`stays` ist leer. Führt Aufgabe 1.1 bis 1.3 noch einmal der Reihe nach aus.")
            return
        if "status" in stays.columns and (stays["status"] != "confirmed").any():
            hint("In `stays` stehen noch stornierte Reservierungen. Es sollen nur bestätigte drin sein.")
            return
        if (stays["nights"] < 1).any():
            hint("Es gibt Aufenthalte mit weniger als einer Nacht. Prüft die Rechnung für `nights`.")
            return
        if not stays["weekday"].between(0, 6).all():
            hint("`weekday` sollte zwischen 0 (Montag) und 6 (Sonntag) liegen.")
            return
        ok(f"`stays` enthält {_block1_number(len(stays))} bestätigte Aufenthalte mit allen neuen Spalten.")
    except Exception as error:
        hint(f"Prüfung nicht möglich: {error}")


def check_avg_nights(avg_nights, stays):
    """Aufgabe 1.3: durchschnittliche Nächte."""
    try:
        if not has_columns(stays, ["nights"], "stays"):
            return
        try:
            value = float(avg_nights)
        except (TypeError, ValueError):
            hint("`avg_nights` sollte eine einzelne Zahl sein. Ruft die Methode mit runden Klammern auf: `.methode()`.")
            return
        nights = stays["nights"]
        if abs(value - nights.mean()) < 0.01:
            shown = f"{value:.2f}".replace(".", ",")
            ok(f"Ein Aufenthalt dauert im Durchschnitt {shown} Nächte.")
            return
        if abs(value - nights.median()) < 0.01:
            hint("Das ist der Median (der mittlere Wert). Gesucht ist der Durchschnitt.")
        elif abs(value - nights.sum()) < 0.5:
            hint("Das ist die Summe aller Nächte. Gesucht ist der Durchschnitt.")
        elif abs(value - nights.max()) < 0.01 or abs(value - nights.min()) < 0.01:
            hint("Das ist ein Extremwert (Maximum oder Minimum). Gesucht ist der Durchschnitt.")
        else:
            hint("Der Wert passt nicht zum Durchschnitt von `stays['nights']`. Schaut ins Mini-Beispiel.")
    except Exception as error:
        hint(f"Prüfung nicht möglich: {error}")


def check_bucket_means(bucket_means):
    """Aufgabe 1.5: Mittelwerte von nights und persons je Vorlauf-Bucket."""
    try:
        if not has_columns(bucket_means, ["nights", "persons"], "bucket_means"):
            return
        if len(bucket_means) == 0 or len(bucket_means) > 5:
            hint("`bucket_means` sollte eine Zeile je Vorlauf-Bucket haben (höchstens 5).")
            return
        nights = pd.to_numeric(bucket_means["nights"], errors="coerce")
        persons = pd.to_numeric(bucket_means["persons"], errors="coerce")
        if nights.isna().any() or persons.isna().any():
            hint("In `bucket_means` stehen leere Werte. Habt ihr `observed=True` stehen lassen?")
            return
        if nights.max() > 30 or persons.max() > 10:
            hint("Die Werte sind viel zu groß für einen einzelnen Aufenthalt. "
                 "Habt ihr summiert oder gezählt statt den Mittelwert gebildet?")
            return
        if nights.min() < 1 or persons.min() < 1:
            hint("Die Werte sind zu klein. Jeder Aufenthalt hat mindestens eine Nacht und eine Person.")
            return
        ok("Mittelwerte je Vorlauf-Bucket berechnet. Vergleicht kurzen und langen Vorlauf!")
    except Exception as error:
        hint(f"Prüfung nicht möglich: {error}")


def check_segment(stays):
    """Aufgabe 1.6: Segmentregel in stays['segment']."""
    try:
        if not isinstance(stays, pd.DataFrame):
            hint("`stays` ist kein DataFrame. Führt Aufgabe 1.3 noch einmal aus.")
            return
        if "segment" not in stays.columns:
            hint("Die Spalte `segment` fehlt noch. Speichert eure Regel in `stays['segment']`.")
            return
        values = set(stays["segment"].dropna().unique())
        if stays["segment"].isna().any():
            hint("In `segment` gibt es leere Werte. Jeder Aufenthalt braucht ein Segment.")
            return
        if not values <= {"business", "leisure"}:
            extra = ", ".join(sorted(str(v) for v in values - {"business", "leisure"}))
            hint(f"`segment` darf nur 'business' und 'leisure' enthalten, gefunden: {extra}.")
            return
        if values != {"business", "leisure"}:
            hint("Es kommt nur ein Segment vor. Eure Grenze ist zu groß oder zu klein.")
            return
        share = (stays["segment"] == "business").mean()
        if share < 0.15:
            hint(f"Nur {share:.0%} business. Eure Grenze ist zu streng. "
                 "Schaut ins Histogramm: Wo liegt das Tal zwischen den Gipfeln?")
            return
        if share > 0.70:
            hint(f"{share:.0%} business ist zu viel. Eure Grenze ist zu großzügig. "
                 "Schaut ins Histogramm: Wo liegt das Tal zwischen den Gipfeln?")
            return
        if "lead_days" in stays.columns:
            lead = stays.groupby("segment")["lead_days"].mean()
            if lead["business"] >= lead["leisure"]:
                hint("Bei euch buchen business-Gäste im Schnitt früher als leisure-Gäste. "
                     "Geschäftsreisende buchen kurzfristig. Stimmt die Richtung eurer Bedingung?")
                return
        ok(f"Segmentregel passt: {share:.0%} business, {1 - share:.0%} leisure.")
    except Exception as error:
        hint(f"Prüfung nicht möglich: {error}")


def check_segment_profile(segment_profile):
    """Aufgabe 1.6: Steckbrief je Segment."""
    try:
        if not has_columns(segment_profile, ["stays", "nights", "persons"], "segment_profile"):
            return
        if not {"business", "leisure"} <= set(segment_profile.index):
            hint("`segment_profile` sollte je eine Zeile für business und leisure haben. Gruppiert nach `segment`.")
            return
        persons = pd.to_numeric(segment_profile["persons"], errors="coerce")
        if persons.isna().any():
            hint("In der Spalte `persons` stehen keine Zahlen. Die Rechenart ist ein Text wie im Mini-Beispiel.")
            return
        if persons.max() > 10:
            hint("`persons` ist viel zu groß für eine Reisegruppe. Habt ihr summiert oder gezählt statt den Mittelwert gebildet?")
            return
        if persons.min() < 1:
            hint("`persons` ist kleiner als 1. Jeder Aufenthalt hat mindestens eine Person.")
            return
        ok("Steckbrief fertig. Vergleicht Vorlauf, Nächte und Personen der beiden Segmente.")
    except Exception as error:
        hint(f"Prüfung nicht möglich: {error}")


# ---- 01b_sql.py ----
# Prüf-Funktionen für Block 1b: SQL und pandas
# Nutzt ok, hint und has_columns aus checks/00_base.py.


def check_same_total(sql_df, pandas_df):
    """Aufgabe 1b.2: SQL- und pandas-Ergebnis zählen dieselben Buchungen."""
    try:
        if not has_columns(sql_df, ["category", "bookings"], "sql_df"):
            hint("Führt die SQL-Zelle aus Aufgabe 1b.1 noch einmal aus.")
            return
        if not has_columns(pandas_df, ["category", "bookings"], "pandas_df"):
            hint("Gruppiert nach der Spalte mit dem Kategorienamen, dann heißt die Spalte `category`.")
            return
        if len(pandas_df) != len(sql_df):
            hint(f"SQL liefert {len(sql_df)} Kategorien, pandas {len(pandas_df)}. "
                 "Gruppiert ihr nach derselben Spalte wie das GROUP BY im SQL?")
            return
        if set(pandas_df["category"]) != set(sql_df["category"]):
            hint("In `pandas_df` stehen andere Gruppen als in `sql_df`. "
                 "Gruppiert nach dem Namen der Zimmerkategorie.")
            return
        sql_total = int(pd.to_numeric(sql_df["bookings"]).sum())
        pandas_total = int(pd.to_numeric(pandas_df["bookings"]).sum())
        if sql_total != pandas_total:
            hint(f"SQL zählt {sql_total} Buchungen, pandas {pandas_total}. "
                 "Wurde die Datenbank seit dem Laden neu befüllt? Dann hilft `tables = load_tables(refresh=True)` "
                 "und Block 1 noch einmal ausführen.")
            return
        ok(f"SQL und pandas kommen auf dieselben {sql_total} Buchungen in {len(sql_df)} Kategorien.")
    except Exception as error:
        hint(f"Prüfung nicht möglich: {error}")


def check_channel_rates(channel_rates):
    """Aufgabe 1b.3: Durchschnittspreis je Vertriebskanal 2024."""
    try:
        if not has_columns(channel_rates, ["channel", "avg_rate", "bookings"], "channel_rates"):
            hint("Die Spaltennamen kommen aus dem SQL: `channel`, `... AS avg_rate`, `... AS bookings`.")
            return
        channels = set(channel_rates["channel"])
        if len(channel_rates) != 3 or channels != {"direct", "corporate", "ota"}:
            hint(f"Erwartet ist genau eine Zeile je Kanal (direct, corporate, ota), gefunden: {len(channel_rates)} Zeilen. "
                 "Wonach muss gruppiert werden, damit jeder Kanal einmal vorkommt?")
            return
        rates = pd.to_numeric(channel_rates["avg_rate"], errors="coerce")
        bookings = pd.to_numeric(channel_rates["bookings"], errors="coerce")
        if rates.isna().any() or (rates <= 0).any():
            hint("`avg_rate` sollte für jeden Kanal ein positiver Preis sein.")
            return
        if bookings.isna().any() or (bookings <= 0).any():
            hint("`bookings` sollte für jeden Kanal eine positive Anzahl sein.")
            return
        if not rates.is_monotonic_decreasing:
            hint("Die Kanäle sind nicht absteigend nach `avg_rate` sortiert. Lasst das ORDER BY stehen.")
            return
        top = channel_rates["channel"].iloc[0]
        ok(f"Drei Kanäle ausgewertet, den höchsten Durchschnittspreis 2024 hat '{top}'.")
    except Exception as error:
        hint(f"Prüfung nicht möglich: {error}")


# ---- 02_kalender.py ----
# Prüf-Funktionen für Block 2: Kalender-Detektive
import pandas as pd


def check_monthly(monthly):
    """Aufgabe 2.1: Monat x Jahr mit Anzahl der Anreisen."""
    try:
        if not isinstance(monthly, pd.DataFrame):
            hint("`monthly` sollte eine Tabelle (DataFrame) sein, so wie `pivot_table` sie liefert.")
            return
        if any(month not in monthly.index for month in range(1, 13)):
            hint("Die Zeilen von `monthly` sollten die Monate 1 bis 12 sein. Prüft `index=`.")
            return
        missing = [year for year in (2023, 2024, 2025) if year not in monthly.columns]
        if missing:
            hint(f"In den Spalten von `monthly` fehlen die Jahre {missing}. Die Spalten sollten Jahre sein.")
            return
        values = monthly.loc[list(range(1, 13)), [2023, 2024, 2025]]
        if values.isna().any().any() or (values <= 0).any().any():
            hint("Für 2023 bis 2025 sollte jeder Monat mehr als 0 Anreisen haben. Zählt ihr mit `aggfunc=\"count\"`?")
            return
        ok("`monthly` hat eine Zeile je Monat und eine Spalte je Jahr.")
    except Exception as error:
        hint(f"`monthly` ließ sich nicht prüfen ({type(error).__name__}). Führt die Zellen davor noch einmal aus.")


def check_event_curve(curve_rel):
    """Aufgabe 2.2: relative Anreisen je Tag rund um den Eventstart."""
    try:
        if not isinstance(curve_rel, pd.DataFrame):
            hint("`curve_rel` sollte eine Tabelle (DataFrame) mit einer Spalte je Event sein.")
            return
        if any(day not in curve_rel.index for day in range(-14, 22)):
            hint("Der Index von `curve_rel` sollte alle Tage von -14 bis 21 enthalten. Prüft die Spalte `offset` und den Filter.")
            return
        if curve_rel.shape[1] < 2:
            hint("`curve_rel` sollte mehrere Events als Spalten haben. Prüft `columns=\"name\"` in der `pivot_table`.")
            return
        before = curve_rel[curve_rel.index.isin(range(-14, 0))].mean().mean()
        if not 0.9 <= before <= 1.1:
            hint("Vor dem Event (Tage -14 bis -1) sollte die Kurve im Mittel bei etwa 1 liegen. "
                 "Habt ihr durch den Normalwert der 14 Tage davor geteilt?")
            return
        peak = curve_rel[curve_rel.index.isin(range(0, 11))].max().max()
        if not peak > 1.2:
            hint("Ab Tag 0 sollte die Kurve deutlich über 1 steigen. Prüft die Richtung von `offset`: "
                 "Anreise minus Eventstart, damit Tag 0 der erste Eventtag ist.")
            return
        ok(f"`curve_rel` deckt die Tage -14 bis 21 ab und enthält {curve_rel.shape[1]} Events.")
    except Exception as error:
        hint(f"`curve_rel` ließ sich nicht prüfen ({type(error).__name__}). Führt die Zellen davor noch einmal aus.")


def check_euro(euro_rel):
    """Aufgabe 2.3: Anreisen Juni/Juli je Gruppe, relativ zu 2023."""
    try:
        if not isinstance(euro_rel, pd.DataFrame):
            hint("`euro_rel` sollte eine Tabelle (DataFrame) mit einer Spalte je Gruppe sein.")
            return
        if not has_columns(euro_rel, ["Austragungsort", "Vergleichsgruppe"], "euro_rel"):
            hint("Beide Gruppen müssen vorkommen. Prüft, ob `isin` die Liste der Austragungsorte bekommt.")
            return
        if 2023 not in euro_rel.index or 2024 not in euro_rel.index:
            hint("Die Zeilen von `euro_rel` sollten die Jahre sein, mindestens 2023 und 2024.")
            return
        base = euro_rel.loc[2023, ["Austragungsort", "Vergleichsgruppe"]]
        if not ((base - 1).abs() < 0.01).all():
            hint("Im Jahr 2023 sollten beide Gruppen genau bei 1 liegen. Habt ihr durch die Zeile 2023 geteilt?")
            return
        ok("`euro_rel` vergleicht beide Gruppen relativ zu 2023.")
    except Exception as error:
        hint(f"`euro_rel` ließ sich nicht prüfen ({type(error).__name__}). Führt die Zellen davor noch einmal aus.")


def check_cancel_rate(cancel_rate):
    """Aufgabe 2.4: Anteil stornierter Reservierungen je Kanal."""
    try:
        if not isinstance(cancel_rate, pd.Series):
            hint("`cancel_rate` sollte eine Series mit einem Wert je Kanal sein.")
            return
        missing = [name for name in ("direct", "corporate", "ota") if name not in cancel_rate.index]
        if missing:
            hint(f"Im Index von `cancel_rate` fehlen die Kanäle {missing}. Gruppiert nach `channel`.")
            return
        if not cancel_rate.between(0, 1).all():
            hint("Eine Quote liegt zwischen 0 und 1. Bildet ihr den Mittelwert einer True/False-Spalte?")
            return
        if (cancel_rate == 0).all() or (cancel_rate == 1).all():
            hint("Alle Quoten sind 0 oder alle sind 1. Welche Werte gibt es in `reservations[\"status\"]`? "
                 "Probiert `reservations[\"status\"].unique()`.")
            return
        ok("`cancel_rate` enthält eine Stornoquote je Kanal.")
    except Exception as error:
        hint(f"`cancel_rate` ließ sich nicht prüfen ({type(error).__name__}). Führt die Zellen davor noch einmal aus.")


# ---- 03_preise.py ----
# Prüf-Funktionen für Block 3: Preisfalle und Umsatzkurve
# Grundfunktionen ok, hint, has_columns kommen aus checks/00_base.py.
import numpy as np
import pandas as pd


def _is_number(value):
    """True für int/float (auch numpy), aber nicht für bool."""
    return pd.api.types.is_number(value) and not isinstance(value, (bool, np.bool_))


def check_daily(daily, daily_rates):
    """Aufgabe 3.1: daily hat je Kategorie und Anreisetag genau eine Zeile, auch ohne Verkauf."""
    try:
        if not isinstance(daily, pd.DataFrame):
            hint("daily sollte ein DataFrame sein.")
            return
        if not has_columns(daily, ["bookings", "r"], "daily"):
            return
        if len(daily) < len(daily_rates):
            hint(f"daily hat {len(daily):,} Zeilen, daily_rates aber {len(daily_rates):,}. "
                 "Es fehlen Tage ohne Verkauf. Welches how behält alle Zeilen der linken Tabelle?")
            return
        if len(daily) > len(daily_rates):
            hint(f"daily hat mehr Zeilen ({len(daily):,}) als daily_rates ({len(daily_rates):,}). "
                 "Habt ihr die Zelle mehrfach ausgeführt? Startet ab dem ersten Merge neu.")
            return
        if daily["bookings"].isna().any():
            hint("In bookings gibt es noch leere Werte (NaN). Ersetzt sie mit fillna.")
            return
        if not (daily["bookings"] == 0).any():
            hint("Es gibt keinen Tag mit 0 Buchungen. Welche Zahl gehört zu einem Tag ohne Verkauf?")
            return
        if daily["r"].isna().any() or not daily["r"].between(0.1, 5).all():
            hint("Einige Werte von r sehen seltsam aus. r ist offered_rate geteilt durch base_price.")
            return
        ok(f"daily ist fertig: {len(daily):,} Tage, davon {(daily['bookings'] == 0).mean():.0%} ohne Verkauf.")
    except Exception as error:
        hint(f"Die Prüfung konnte nicht laufen: {error}")


def check_day_type(regular):
    """Aufgabe 3.3: Tagesart Event > Sommer > Alltag, passend zu is_event und Monat."""
    try:
        if not isinstance(regular, pd.DataFrame):
            hint("regular sollte ein DataFrame sein.")
            return
        if not has_columns(regular, ["arrival_date", "is_event", "day_type"], "regular"):
            return
        if regular["is_event"].dtype != bool:
            hint("is_event sollte nur True/False enthalten. Habt ihr .notna() ausgeführt?")
            return
        allowed = {"Event", "Sommer", "Alltag"}
        found = set(regular["day_type"].dropna().unique())
        if found - allowed:
            hint(f"Unerwartete Werte in day_type: {sorted(found - allowed)}. "
                 "Erlaubt sind genau 'Event', 'Sommer' und 'Alltag'.")
            return
        missing = allowed - found
        if missing:
            hint(f"Diese Tagesarten kommen gar nicht vor: {sorted(missing)}. Prüft eure Bedingungen.")
            return
        day_type = regular["day_type"]
        month = regular["arrival_date"].dt.month
        summer_month = month.isin([6, 7, 8])
        if (regular["is_event"] & (day_type != "Event")).any():
            hint("Einige Event-Tage sind nicht als 'Event' markiert. "
                 "Die Event-Bedingung gehört ins äußere np.where.")
            return
        if ((day_type == "Event") & ~regular["is_event"]).any():
            hint("Einige Tage ohne Event sind als 'Event' markiert. Prüft die äußere Bedingung.")
            return
        if ((day_type == "Sommer") & ~summer_month).any():
            hint("Einige 'Sommer'-Tage liegen nicht im Sommer. Sommer sind die Monate Juni bis August.")
            return
        if ((day_type == "Alltag") & summer_month).any():
            hint("Einige Sommertage sind als 'Alltag' markiert. Prüft die Monatsliste und die innere Bedingung.")
            return
        ok("Jeder Tag hat die richtige Tagesart: Event, Sommer oder Alltag.")
    except Exception as error:
        hint(f"Die Prüfung konnte nicht laufen: {error}")


def check_closed_share(share_closed):
    """Aufgabe 3.4: Anteil der geschlossenen Tage als Zahl zwischen 0 und 1."""
    try:
        if not _is_number(share_closed):
            hint("share_closed sollte eine einzelne Zahl sein, kein DataFrame und keine Series.")
            return
        if share_closed > 1:
            hint("Der Wert ist größer als 1. Das ist eine Anzahl, kein Anteil. "
                 "Welche Funktion liefert bei True/False den Anteil?")
            return
        if share_closed <= 0:
            hint("Der Anteil ist 0 oder kleiner. Habt ihr die Spalte closed_to_arrival von daily benutzt?")
            return
        ok(f"An {share_closed:.1%} der Tage war keine Anreise mehr möglich.")
    except Exception as error:
        hint(f"Die Prüfung konnte nicht laufen: {error}")


def check_test_days(test):
    """Aufgabe 3.5: test enthält nur Tage mit zufälligem Preistest."""
    try:
        if not isinstance(test, pd.DataFrame):
            hint("test sollte ein DataFrame sein.")
            return
        if not has_columns(test, ["price_source", "r", "bookings"], "test"):
            return
        if len(test) == 0:
            hint("test ist leer. Steht der Wert im query-Text in einfachen Anführungszeichen "
                 "und genau so geschrieben wie in price_source?")
            return
        sources = set(test["price_source"].unique())
        if sources != {"test"}:
            hint(f"In test stecken noch diese Preisquellen: {sorted(sources)}. "
                 "Behaltet nur die Tage mit zufälligem Preistest.")
            return
        ok(f"test enthält nur Preistest-Tage ({len(test):,} Zeilen).")
    except Exception as error:
        hint(f"Die Prüfung konnte nicht laufen: {error}")


def check_revenue_peak(best_r):
    """Aufgabe 3.6: Gipfel des Umsatzbergs liegt in einem plausiblen Bereich."""
    try:
        if not _is_number(best_r):
            hint("best_r sollte eine einzelne Zahl sein, nämlich eine Preisstufe r.")
            return
        if 0.75 <= best_r <= 1.0:
            ok(f"Der Umsatzberg hat seinen Gipfel bei r = {best_r:.2f}.")
            return
        hint(f"best_r = {best_r} passt noch nicht. Habt ihr die Stelle des Maximums (idxmax) "
             "genommen und nicht das Maximum selbst (max)? Und rechnet ihr nur mit den Testtagen?")
    except Exception as error:
        hint(f"Die Prüfung konnte nicht laufen: {error}")


def check_segment_peaks(peaks):
    """Aufgabe 3.7: Leisure hat den Umsatzgipfel bei niedrigerem r als Business."""
    try:
        if not isinstance(peaks, pd.Series):
            hint("peaks sollte eine Series sein: je Gästegruppe eine Preisstufe r.")
            return
        missing = [name for name in ["business", "leisure"] if name not in peaks.index]
        if missing:
            hint(f"In peaks fehlen die Gruppen {missing}. Gehört segment in columns der pivot_table?")
            return
        business = peaks["business"]
        leisure = peaks["leisure"]
        if not (_is_number(business) and _is_number(leisure)):
            hint("Die Werte in peaks sollten Preisstufen r sein. Nehmt idxmax, nicht max.")
            return
        if not (0.5 <= leisure <= 1.6 and 0.5 <= business <= 1.6):
            hint("Die Werte in peaks sehen nicht wie Preisstufen r aus. Nehmt idxmax, nicht max.")
            return
        if leisure < business:
            ok(f"Zwei Gipfel: leisure bei r = {leisure:.2f}, business bei r = {business:.2f}.")
            return
        hint("Bei euch liegen beide Gipfel gleich oder leisure höher als business. "
             "Prüft eure Segmentregel aus Block 1 und ob ihr mit dem Preis r multipliziert habt.")
    except Exception as error:
        hint(f"Die Prüfung konnte nicht laufen: {error}")


def check_net(test_stays):
    """Aufgabe 3.8: gross und net passen zu Preis, Nächten und Provision."""
    try:
        if not isinstance(test_stays, pd.DataFrame):
            hint("test_stays sollte ein DataFrame sein.")
            return
        needed = ["rate_per_night", "nights", "commission_pct", "gross", "net"]
        if not has_columns(test_stays, needed, "test_stays"):
            return
        data = test_stays[needed].dropna()
        data = data[data["gross"] > 0]
        if len(data) == 0:
            hint("In test_stays gibt es keine Zeilen mit Bruttoumsatz.")
            return
        if not np.allclose(data["gross"], data["rate_per_night"] * data["nights"]):
            hint("gross passt nicht zu Preis je Nacht und Zahl der Nächte.")
            return
        kept = data["net"] / data["gross"]
        lost = 1 - kept
        rate = data["commission_pct"] / 100
        if np.allclose(lost, rate, atol=1e-6):
            ok("net stimmt: Dem Hotel bleibt der Bruttoumsatz abzüglich Provision.")
            return
        if (data["net"] > data["gross"] + 1e-6).any():
            hint("net ist teilweise größer als gross. Die Provision wird abgezogen, nicht addiert.")
            return
        if np.allclose(lost, data["commission_pct"], atol=1e-6):
            hint("Die Provision ist in Prozent angegeben. Teilt ihr durch 100?")
            return
        if np.allclose(kept, rate, atol=1e-6):
            hint("net ist gerade die Provision selbst. Gesucht ist, was dem Hotel übrig bleibt.")
            return
        hint("net passt noch nicht. Bei 0 % Provision muss net gleich gross sein, "
             "bei 20 % Provision bleiben 80 % von gross.")
    except Exception as error:
        hint(f"Die Prüfung konnte nicht laufen: {error}")


# ---- 04_spiegelmining.py ----
# Prüf-Funktionen für Block 4: SpiegelMining im Hotel
# Nutzt ok, hint und has_columns aus checks/00_base.py.


def check_guest_stays(guest_stays):
    if not hasattr(guest_stays, "columns"):
        hint("guest_stays sollte ein DataFrame sein.")
        return
    if not has_columns(guest_stays, ["guest_id", "hotel_id", "city_id", "checkin"], "guest_stays"):
        return
    if len(guest_stays) == 0:
        hint("guest_stays ist leer. Ist die Mindestzahl vielleicht zu hoch?")
        return
    smallest = guest_stays.groupby("guest_id").size().min()
    if smallest < 4:
        hint(f"Es gibt noch Gäste mit nur {smallest} Aufenthalten. Behaltet nur Gäste mit mindestens 4.")
        return
    if smallest > 4:
        hint("Der kleinste Gast hat mehr als 4 Aufenthalte. Habt ihr mit > statt >= oder mit einer zu großen Zahl gefiltert?")
        return
    ok(f"guest_stays enthält {guest_stays['guest_id'].nunique():,} Gäste mit mindestens 4 Aufenthalten.")


def check_pairs(pairs):
    if not hasattr(pairs, "columns"):
        hint("pairs sollte ein DataFrame sein.")
        return
    if not has_columns(pairs, ["a", "b", "hotel_id", "checkin"], "pairs"):
        return
    if len(pairs) == 0:
        hint("pairs ist leer. Merged ihr auf Hotel und Anreisetag?")
        return
    if (pairs["a"] == pairs["b"]).any():
        hint("pairs enthält noch Selbstpaare (a == b). Habt ihr query('a < b') angewendet?")
        return
    if not (pairs["a"] < pairs["b"]).all():
        hint("Es gibt Zeilen mit a > b. Damit zählt ihr jedes Paar doppelt.")
        return
    ok(f"pairs enthält {len(pairs):,} gemeinsame Anreisen, jedes Paar nur in einer Richtung.")


def check_top_pairs(top_pairs):
    if not hasattr(top_pairs, "is_monotonic_decreasing") or hasattr(top_pairs, "columns"):
        hint("top_pairs sollte eine Series sein: das Ergebnis von groupby(...).size().")
        return
    if len(top_pairs) == 0:
        hint("top_pairs ist leer.")
        return
    if len(top_pairs) > 10:
        hint("Zeigt nur die 10 häufigsten Paare (head(10)).")
        return
    if not top_pairs.is_monotonic_decreasing:
        hint("Die Paare sind nicht absteigend sortiert. Die häufigsten Paare gehören nach oben.")
        return
    ok(f"Die 10 häufigsten Paare stehen oben, Spitzenreiter mit {top_pairs.iloc[0]} gemeinsamen Anreisen.")


def check_candidates(candidates, pairs):
    if not hasattr(candidates, "columns"):
        hint("candidates sollte ein DataFrame sein.")
        return
    if not has_columns(candidates, ["a", "b"], "candidates"):
        return
    if len(candidates) == 0:
        hint("candidates ist leer. Ist der Filter zu streng?")
        return
    if candidates.duplicated(["a", "b"]).any():
        hint("Manche Paare kommen mehrfach vor. candidates sollte jedes Paar nur einmal enthalten.")
        return
    try:
        enough = pairs.groupby(["a", "b"])["hotel_id"].nunique()
    except Exception:
        ok(f"candidates enthält {len(candidates):,} Paare.")
        return
    expected = int((enough >= 4).sum())
    if len(candidates) > expected:
        hint("candidates enthält Paare mit weniger als 4 verschiedenen Hotels. Zählt ihr wirklich verschiedene Hotels?")
        return
    if len(candidates) < expected:
        hint("candidates enthält zu wenige Paare. Filtert auf mindestens 4 verschiedene Hotels (>=).")
        return
    ok(f"candidates enthält {len(candidates):,} Paare, die sich in mindestens 4 Hotels treffen.")


def check_result(result):
    if not hasattr(result, "columns"):
        hint("result sollte ein DataFrame sein.")
        return
    if not has_columns(result, ["a", "b", "meetings", "hotels"], "result"):
        return
    if len(result) == 0:
        hint("result ist leer. Prüft den Anti-Join: Bleiben die Zeilen mit _merge == 'left_only'?")
        return
    if (result["hotels"] < 4).any():
        hint("result enthält Paare mit weniger als 4 verschiedenen Hotels.")
        return
    if not result["meetings"].is_monotonic_decreasing:
        hint("result ist nicht absteigend nach meetings sortiert.")
        return
    if len(result) > 50:
        hint(f"result hat noch {len(result):,} Paare. Sind die Messetage wirklich herausgefiltert?")
        return
    ok(f"result enthält {len(result)} auffällige Paare. Schaut euch die obersten genau an.")


def check_lag_stats(lag_stats):
    if not hasattr(lag_stats, "columns"):
        hint("lag_stats sollte ein DataFrame sein.")
        return
    if not has_columns(lag_stats, ["leader", "follower", "trips", "hotels"], "lag_stats"):
        return
    if len(lag_stats) == 0:
        hint("lag_stats ist leer. Habt ihr das Datum um einen Tag verschoben?")
        return
    if (lag_stats["leader"] == lag_stats["follower"]).any():
        hint("lag_stats enthält Selbstpaare (leader == follower).")
        return
    if not lag_stats["trips"].is_monotonic_decreasing:
        hint("lag_stats ist nicht absteigend nach trips sortiert.")
        return
    ok(f"lag_stats enthält {len(lag_stats)} Paare mit versetzter Anreise in mindestens 4 Hotels.")

## Block 1: Wer sind unsere Gäste?

In unserer Hoteldatenbank stecken Privat- und Geschäftsreisende. Für das Hotel ist das ein großer Unterschied: Geschäftsreisende brauchen ein Zimmer am Montag in der Messestadt, Urlauber*innen ein Familienzimmer am Wochenende an der See. Leider gibt es **keine Spalte**, die verrät, wer wer ist.

In diesem Block lernt ihr die wichtigsten pandas-Werkzeuge kennen und findet am Ende selbst eine Regel, die die beiden Gruppen trennt. Schritt für Schritt:

1. Daten laden und anschauen
2. Tabellen verbinden (wie `JOIN` in SQL)
3. Eine Arbeitstabelle `stays` bauen
4. Den Buchungsvorlauf anschauen
5. Gruppen bilden und vergleichen
6. Eure eigene Segmentregel

### Aufgabe 1.1: Daten laden und anschauen · ⏱ ca. 10 Min.

Die Setup-Zelle oben hat die Funktion `load_tables()` bereitgestellt. Sie holt alle Tabellen aus der Datenbank und gibt sie als **Dictionary** zurück: Zu jedem Tabellennamen gehört ein **DataFrame**.

Ein DataFrame ist eine Tabelle in Python: Zeilen, Spalten mit Namen, und jede Spalte hat einen Datentyp (Zahl, Text, Datum, …). Im Grunde dasselbe wie eine Tabelle in der Datenbank, nur dass sie jetzt im Arbeitsspeicher liegt und wir mit Python darin rechnen.

In [ ]:
tables = load_tables()
check_tables(tables)

Damit wir nicht immer `tables["reservations"]` schreiben müssen, geben wir jeder Tabelle einen kurzen Namen. Diese Namen benutzen wir im ganzen Notebook.

In [ ]:
cities = tables["cities"]
hotels = tables["hotels"]
categories = tables["room_categories"]
guests = tables["guests"]
reservations = tables["reservations"]
daily_rates = tables["daily_rates"]
events = tables["calendar_events"]

Vier Werkzeuge, um einen DataFrame kennenzulernen:

| Befehl | Was es zeigt |
|---|---|
| `df.head()` | die ersten 5 Zeilen (`df.head(10)`: die ersten 10) |
| `df.shape` | (Anzahl Zeilen, Anzahl Spalten) |
| `len(df)` | nur die Anzahl Zeilen |
| `df.info()` | alle Spalten mit Datentyp und Anzahl gefüllter Werte |

**Mini-Beispiel:** Ein Spielzeug-DataFrame mit drei Gästen.

In [ ]:
toy = pd.DataFrame({
    "name": ["Ada", "Ben", "Cem"],
    "nights": [2, 5, 1],
})
print("shape:", toy.shape)        # (3, 2): 3 Zeilen, 2 Spalten
print("Zeilen:", toy.shape[0])    # Position 0 im Paar = Zeilen
print("Spalten:", toy.shape[1])   # Position 1 im Paar = Spalten
print("len:", len(toy))           # ebenfalls 3
toy.head()

Jetzt die echte Tabelle `reservations`. Die letzte Zeile einer Zelle wird automatisch angezeigt.

In [ ]:
reservations.head()

In [ ]:
reservations.info()

Achtet in der Ausgabe von `info()` auf die Spalte `Dtype`: `int64` sind ganze Zahlen, `float64` Kommazahlen, `object` meist Text und `datetime64[ns]` Datum mit Uhrzeit. Mit Datumsspalten kann pandas rechnen, das brauchen wir gleich.

Jetzt seid ihr dran. Eine Lücke ist immer `___` (drei Unterstriche). Die Meldung `LueckeOffen: In dieser Zelle steht noch eine Lücke ___` heißt: Hier fehlt noch eure Lösung.

In [ ]:
# TODO: Wie viele Reservierungen gibt es? Und wie viele Spalten hat die Tabelle?
n_reservations = len(___)
n_columns = reservations.shape[___]
print(f"{n_reservations:,} Reservierungen mit je {n_columns} Spalten")

In [ ]:
check_counts(n_reservations, n_columns, reservations)

### Aufgabe 1.2: Tabellen verbinden · ⏱ ca. 15 Min.

Eine Reservierung kennt nur ihre `category_id`. In welchem Hotel und in welcher Stadt das Zimmer liegt, steht in anderen Tabellen:

`reservations.category_id` → `room_categories.id` → `room_categories.hotel_id` → `hotels.id` → `hotels.city_id` → `cities.id`

In SQL würdet ihr dafür `JOIN … ON …` schreiben. In pandas heißt das **`merge`**:

| SQL | pandas |
|---|---|
| `FROM a JOIN b ON a.key = b.key` | `a.merge(b, on="key")` |

`on=` nennt die Spalte, die in **beiden** Tabellen gleich heißt. Wie beim SQL-`JOIN` (ein *inner join*) bleiben nur Zeilen, die in beiden Tabellen einen Partner finden.

**Mini-Beispiel:** Zwei Spielzeug-Tabellen, verbunden über `city_id`.

In [ ]:
toy_hotels = pd.DataFrame({
    "hotel": ["Seeblick", "Messehof", "Alpenruh"],
    "city_id": [1, 2, 3],
})
toy_cities = pd.DataFrame({
    "city_id": [1, 2, 3],
    "city": ["Sylt", "Hannover", "Garmisch-Partenkirchen"],
})
toy_hotels.merge(toy_cities, on="city_id")

Ein Problem: In unseren echten Tabellen heißt der Schlüssel oft einfach `id`, und auch `name` gibt es mehrfach (Kategorie-Name, Hotel-Name, Stadt-Name). Deshalb benennen wir Spalten vor dem `merge` um. Das macht **`rename`**, ähnlich wie `AS` in SQL.

Mit doppelten eckigen Klammern `df[["a", "b"]]` wählt ihr Spalten aus, wie die Spaltenliste hinter `SELECT`.

**Mini-Beispiel:**

In [ ]:
toy_rooms = pd.DataFrame({"id": [7, 8], "name": ["Standard", "Suite"], "base_price": [90.0, 250.0]})
toy_rooms.rename(columns={"id": "category_id", "name": "category"})[["category_id", "category"]]

Jetzt bauen wir die Tabelle `places`: **eine Zeile je Zimmerkategorie** mit Hotel, Stadt, Stadttyp (`city` = Großstadt, `resort` = Urlaubsort) und Bundesland. Diese Zelle ist vollständig vorgegeben, weil alle späteren Blöcke `places` brauchen. Lest sie trotzdem Zeile für Zeile.

In [ ]:
# Schritt 1: Spalten umbenennen, damit die Schlüssel zusammenpassen
rooms = categories.rename(columns={"id": "category_id", "name": "category"})
houses = hotels.rename(columns={"id": "hotel_id"})[["hotel_id", "city_id", "city_type", "bundesland"]]
towns = cities.rename(columns={"id": "city_id", "name": "city"})[["city_id", "city"]]

# Schritt 2: zwei JOINs hintereinander
places = rooms.merge(houses, on="hotel_id").merge(towns, on="city_id")
places.head()

**Kleine Übung:** Welches Hotel liegt in welcher Stadt? Verbindet `hotels` mit `towns`. Überlegt, über welche Spalte die beiden Tabellen zusammengehören. (Tipp: Sie steht in beiden Tabellen und heißt gleich.)

In [ ]:
# TODO: Ergänzt die Schlüsselspalte für den merge (als Text in Anführungszeichen).
hotel_cities = hotels.merge(towns, on=___)
hotel_cities[["name", "city", "city_type", "stars"]]

In [ ]:
check_hotel_cities(hotel_cities, hotels)

Wie viele unserer Hotels liegen in Großstädten, wie viele an Urlaubsorten? **`value_counts()`** zählt, wie oft jeder Wert in einer Spalte vorkommt.

**Mini-Beispiel:**

In [ ]:
toy_types = pd.DataFrame({"city_type": ["city", "resort", "city", "city"]})
toy_types["city_type"].value_counts()

In [ ]:
hotel_cities["city_type"].value_counts()

### Aufgabe 1.3: Die Arbeitstabelle `stays` · ⏱ ca. 15 Min.

Für unsere Auswertungen zählen nur **bestätigte** Aufenthalte, Stornierungen lassen wir weg. Zeilen filtern geht mit **`query`**: Ihr schreibt die Bedingung wie hinter `WHERE` in SQL, aber in Anführungszeichen. Texte stehen darin in einfachen Anführungszeichen `'…'`.

**Mini-Beispiel:**

In [ ]:
toy_bookings = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "status": ["confirmed", "cancelled", "confirmed", "confirmed"],
    "adults": [1, 2, 2, 1],
})
toy_bookings.query("status == 'confirmed'")

Mit Datumsspalten kann pandas rechnen. Zieht man zwei Datumswerte voneinander ab, entsteht eine Zeitspanne. Über **`.dt`** kommt ihr an die Teile eines Datums oder einer Zeitspanne:

| Befehl | Ergebnis |
|---|---|
| `(ende - anfang).dt.days` | Zeitspanne in ganzen Tagen |
| `datum.dt.day_of_week` | Wochentag, 0 = Montag … 6 = Sonntag |
| `datum.dt.month`, `datum.dt.year` | Monat, Jahr |
| `datum.dt.normalize()` | Uhrzeit auf 00:00 setzen, nur der Tag zählt |

**Mini-Beispiel:**

In [ ]:
toy_dates = pd.DataFrame({
    "checkin": pd.to_datetime(["2024-03-04", "2024-03-08", "2024-12-24"]),
    "checkout": pd.to_datetime(["2024-03-06", "2024-03-10", "2024-12-27"]),
})
toy_dates["nights"] = (toy_dates["checkout"] - toy_dates["checkin"]).dt.days
toy_dates["weekday"] = toy_dates["checkin"].dt.day_of_week
toy_dates["month"] = toy_dates["checkin"].dt.month
toy_dates

Jetzt bauen wir `stays`. Auch diese Zelle ist vollständig vorgegeben, weil spätere Blöcke sie brauchen. Neu ist der **Vorlauf** `lead_days`: die Zahl der Tage zwischen Buchung (`created_at`) und Anreise (`checkin`).

In [ ]:
# Nur bestätigte Reservierungen, dazu Hotel und Stadt aus places
stays = reservations.query("status == 'confirmed'").merge(places, on="category_id")

# Neue Spalten berechnen
stays["nights"] = (stays["checkout"] - stays["checkin"]).dt.days
stays["persons"] = stays["adults"] + stays["children"]
stays["lead_days"] = (stays["checkin"] - stays["created_at"].dt.normalize()).dt.days
stays["weekday"] = stays["checkin"].dt.day_of_week
stays["month"] = stays["checkin"].dt.month
stays["year"] = stays["checkin"].dt.year

stays[["checkin", "checkout", "nights", "persons", "lead_days", "weekday", "city"]].head()

In [ ]:
check_stays(stays)

**Kleine Übung:** Wie viele Nächte bleibt ein Gast im Durchschnitt? Eine Spalte lässt sich mit einer Methode zu einer einzigen Zahl zusammenfassen.

**Mini-Beispiel:**

In [ ]:
toy_nights = pd.Series([2, 5, 1, 4])
print("Summe:     ", toy_nights.sum())
print("Mittelwert:", toy_nights.mean())
print("Median:    ", toy_nights.median())
print("Maximum:   ", toy_nights.max())

In [ ]:
# TODO: Berechnet den Mittelwert der Spalte nights.
avg_nights = stays["nights"].___()
print(f"Im Durchschnitt {avg_nights:.2f} Nächte")

In [ ]:
check_avg_nights(avg_nights, stays)

### Aufgabe 1.4: Wie weit im Voraus wird gebucht? · ⏱ ca. 5 Min.

Ein **Histogramm** zeigt, wie sich die Werte einer Spalte verteilen: Die x-Achse wird in schmale Bereiche geteilt, die Höhe eines Balkens ist die Anzahl der Werte darin.

Ganz wenige Buchungen haben einen sehr langen Vorlauf. Damit die Achse lesbar bleibt, setzt `clip(upper=150)` alle Werte über 150 auf 150. Diese Zelle ist vorgegeben, ihr müsst nur hinschauen.

In [ ]:
stays["lead_days"].clip(upper=150).plot.hist(bins=50, title="Vorlauf in Tagen (gekappt bei 150)")
plt.xlabel("Tage zwischen Buchung und Anreise")
plt.show()

🤔 Wie viele Gipfel seht ihr? Was könnte hinter jedem Gipfel stecken? Wo liegt ungefähr das Tal dazwischen?

### Aufgabe 1.5: Vorlauf in Gruppen · ⏱ ca. 15 Min.

Mit vielen einzelnen Tageswerten lässt sich schlecht vergleichen. Wir teilen den Vorlauf deshalb in **Bereiche** („Buckets“) ein. Das macht **`pd.cut`**:

- `bins=` sind die Grenzen. Ein Bereich enthält die rechte Grenze, aber nicht die linke: Zwischen −1 und 14 liegen also die Werte 0 bis 14.
- `labels=` sind die Namen der Bereiche.

**Mini-Beispiel:**

In [ ]:
bins = [-1, 14, 29, 60, 120, 10_000]
labels = ["0-14", "15-29", "30-60", "61-120", "über 120"]
toy_lead = pd.DataFrame({"lead_days": [0, 14, 15, 45, 200]})
toy_lead["lead_bucket"] = pd.cut(toy_lead["lead_days"], bins=bins, labels=labels)
toy_lead

Jetzt für alle Aufenthalte (vorgegeben, spätere Blöcke brauchen `lead_bucket`):

In [ ]:
stays["lead_bucket"] = pd.cut(stays["lead_days"], bins=bins, labels=labels)
stays["lead_bucket"].value_counts().sort_index()

**An welchen Wochentagen reisen die Gäste aus den einzelnen Buckets an?** Dafür brauchen wir eine Kreuztabelle: Zeilen = Wochentag, Spalten = Vorlauf-Bucket, in jeder Zelle die Anzahl Anreisen. Das macht **`pivot_table`**:

- `index=` bestimmt die Zeilen, `columns=` die Spalten,
- `values=` ist die Spalte, die ausgewertet wird,
- `aggfunc=` sagt wie: `"count"` zählt, `"sum"` summiert, `"mean"` bildet den Mittelwert.

`observed=True` heißt: nur Kombinationen zeigen, die wirklich vorkommen. Das braucht man bei Bucket-Spalten wie `lead_bucket`.

**Mini-Beispiel:**

In [ ]:
toy_arrivals = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "weekday": [0, 0, 4, 4, 4],
    "lead_bucket": ["0-14", "0-14", "0-14", "30-60", "30-60"],
})
toy_arrivals.pivot_table(index="weekday", columns="lead_bucket", values="id", aggfunc="count")

Lest die Tabelle: Am Montag (0) gab es zwei kurzfristige Anreisen, am Freitag (4) eine kurzfristige und zwei mit 30–60 Tagen Vorlauf. Fehlende Kombinationen erscheinen als `NaN` (kein Wert).

Jetzt mit den echten Daten, als gestapeltes Balkendiagramm. Mit `rename` ersetzen wir die Zahlen 0–6 durch Wochentagsnamen.

In [ ]:
weekday_names = {0: "Mo", 1: "Di", 2: "Mi", 3: "Do", 4: "Fr", 5: "Sa", 6: "So"}
table = stays.pivot_table(index="weekday", columns="lead_bucket", values="id", aggfunc="count", observed=True)
table = table.rename(index=weekday_names)
table.plot.bar(stacked=True, title="Anreisen je Wochentag und Vorlauf")
plt.xlabel("Wochentag der Anreise")
plt.ylabel("Anreisen")
plt.show()

Bleiben kurzfristige Gäste kürzer? Reisen sie mit weniger Personen? Dafür bilden wir **Gruppen** und berechnen je Gruppe einen Mittelwert. Das macht **`groupby`**, genau wie `GROUP BY` in SQL:

`df.groupby("gruppenspalte")[["spalte_a", "spalte_b"]].mean()`

**Mini-Beispiel:**

In [ ]:
toy_groups = pd.DataFrame({
    "lead_bucket": ["0-14", "0-14", "30-60", "30-60", "30-60"],
    "nights": [1, 2, 5, 7, 6],
    "persons": [1, 1, 2, 4, 3],
})
toy_groups.groupby("lead_bucket")[["nights", "persons"]].mean()

In [ ]:
# TODO: Berechnet je Vorlauf-Bucket den Mittelwert von nights und persons.
bucket_means = stays.groupby("lead_bucket", observed=True)[["nights", "persons"]].___().round(2)
bucket_means

In [ ]:
check_bucket_means(bucket_means)

🤔 Vergleicht die kurzfristigen Buckets mit den langfristigen: Wochentag der Anreise, Nächte, Personen. Welche zwei Gästetypen erkennt ihr?

### Aufgabe 1.6: Eure Segmentregel · ⏱ ca. 20 Min.

Jetzt legt ihr fest, wer als Geschäftsreise (`"business"`) und wer als Urlaub (`"leisure"`) zählt. Die neue Spalte `stays["segment"]` brauchen spätere Blöcke, zum Beispiel für getrennte Preiskurven.

Eine Wenn-dann-Regel für eine ganze Spalte schreibt man mit **`np.where`**:

`np.where(bedingung, wert_wenn_wahr, wert_wenn_falsch)`

Das entspricht `CASE WHEN bedingung THEN … ELSE … END` in SQL.

**Mini-Beispiel:** Wer mehr als 3 Nächte bleibt, ist ein „Langbleiber“.

In [ ]:
toy_rule = pd.DataFrame({"nights": [1, 4, 2, 7]})
toy_rule["stay_type"] = np.where(toy_rule["nights"] > 3, "long", "short")
toy_rule

Welche Spalte trennt die beiden Gruppen am deutlichsten? Schaut noch einmal auf das Histogramm aus Aufgabe 1.4: Wo liegt das Tal zwischen den beiden Gipfeln? Wählt dort eure Grenze in Tagen.

In [ ]:
# TODO: Ergänzt die Grenze in Tagen. Kurzfristige Buchungen zählen als business.
stays["segment"] = np.where(stays["lead_days"] <= ___, "business", "leisure")
stays["segment"].value_counts(normalize=True).round(2)

In [ ]:
check_segment(stays)

Prüfen wir die Regel: An welchen Wochentagen reisen die beiden Segmente an? Damit große und kleine Gruppen vergleichbar sind, teilen wir jede Spalte durch ihre Summe. So steht in jeder Zelle der **Anteil** der Anreisen (vorgegeben).

In [ ]:
weekday_counts = stays.pivot_table(index="weekday", columns="segment", values="id", aggfunc="count")
weekday_share = (weekday_counts / weekday_counts.sum()).rename(index=weekday_names)
weekday_share.plot.bar(title="Anteil der Anreisen je Wochentag und Segment")
plt.xlabel("Wochentag der Anreise")
plt.ylabel("Anteil")
plt.show()

Zum Schluss ein Steckbrief je Segment. Mit **`agg`** berechnet ihr mehrere Kennzahlen auf einmal. Jede Kennzahl bekommt einen Namen und ein Paar aus Spalte und Rechenart:

`name=("spalte", "rechenart")`, zum Beispiel `nights=("nights", "mean")`.

**Mini-Beispiel:**

In [ ]:
toy_profile = pd.DataFrame({
    "segment": ["business", "business", "leisure", "leisure", "leisure"],
    "nights": [1, 2, 5, 7, 6],
    "persons": [1, 1, 2, 4, 3],
})
toy_profile.groupby("segment").agg(
    stays=("nights", "count"),
    nights=("nights", "mean"),
    max_persons=("persons", "max"),
)

In [ ]:
# TODO: Ergänzt die Rechenart für persons: Wie viele Personen reisen im Durchschnitt mit?
segment_profile = stays.groupby("segment").agg(
    stays=("id", "count"),
    lead_days=("lead_days", "mean"),
    nights=("nights", "mean"),
    persons=("persons", ___),
).round(2)
segment_profile

In [ ]:
check_segment_profile(segment_profile)

🤔 Passt euer Steckbrief zu dem, was ihr von Geschäfts- und Urlaubsreisen erwartet? Wo könnte eure Regel danebenliegen, zum Beispiel bei einer Familie, die spontan ans Meer fährt?

## Block 1b: SQL und pandas – direkt in der Datenbank

Bisher haben wir ganze Tabellen mit `load_tables()` geladen und dann in pandas gerechnet. Man kann die Datenbank aber auch **selbst rechnen lassen** und nur das fertige Ergebnis abholen. Bei großen Tabellen ist das oft schneller, weil nur wenige Zeilen über das Netz kommen.

Dafür steht euch aus der Setup-Zelle die Verbindung **`engine`** zur Verfügung. Ein paar Spielregeln:

- Eure Verbindung darf **nur lesen**. `INSERT`, `UPDATE`, `DELETE` oder `CREATE` werden abgelehnt.
- Alle Tabellen liegen im Schema **`analytics`**. Schreibt deshalb immer `analytics.` vor den Tabellennamen, also `analytics.reservations`, `analytics.hotels` usw. Die Spalten sind dieselben wie in den DataFrames.
- Jede Abfrage darf höchstens **60 Sekunden** laufen, danach bricht die Datenbank ab.
- Bitte **keine riesigen Ergebnisse** abholen (also kein `SELECT * FROM analytics.reservations`). Lasst die Datenbank zählen, summieren und gruppieren.

### Aufgabe 1b.1: Eine Abfrage aus Python · ⏱ ca. 10 Min.

Mit **`pd.read_sql(text(sql), engine)`** schickt ihr eine SQL-Abfrage an die Datenbank und bekommt das Ergebnis als DataFrame zurück. `text(...)` markiert den String als SQL.

**Mini-Beispiel:** Wie viele Hotels gibt es? (Die dreifachen Anführungszeichen `"""` erlauben SQL über mehrere Zeilen.)

In [ ]:
pd.read_sql(text("""
    SELECT COUNT(*) AS n_hotels
    FROM analytics.hotels
"""), engine)

Oft soll ein Wert aus Python in die Abfrage, zum Beispiel eine Hotel-ID. Dafür schreibt ihr im SQL einen **Platzhalter** mit Doppelpunkt (`:min_stars`) und übergebt den Wert getrennt mit `params={...}`.

Warum nicht einfach mit einem f-String (`f"... WHERE stars >= {x}"`) in den Text kleben? Weil dann jeder Text, der in `x` steht, als SQL ausgeführt wird. Kommt dieser Text von außen, etwa aus einem Suchfeld, kann jemand eigenen SQL-Code einschleusen: **SQL-Injection**. Mit Platzhaltern übergibt die Datenbank-Bibliothek den Wert sicher als reinen Wert.

**Mini-Beispiel:**

In [ ]:
pd.read_sql(
    text("""
        SELECT name, stars
        FROM analytics.hotels
        WHERE stars >= :min_stars
        ORDER BY stars DESC
    """),
    engine,
    params={"min_stars": 4},
)

Jetzt eine echte Frage: **Wie oft wurde jede Zimmerkategorie im ersten Hotel gebucht?** Wir verbinden `room_categories` und `reservations` mit `JOIN`, filtern auf ein Hotel und auf bestätigte Buchungen und zählen je Kategorie.

`hotels["id"].iloc[0]` ist die ID in der ersten Zeile von `hotels`. `int(...)` macht daraus eine normale Python-Zahl, denn mit dem pandas-Zahlentyp kann die Datenbank-Bibliothek nichts anfangen. Diese Zelle ist vorgegeben.

In [ ]:
selected_hotel = int(hotels["id"].iloc[0])

sql_df = pd.read_sql(
    text("""
        SELECT c.name AS category, COUNT(*) AS bookings
        FROM analytics.room_categories AS c
        JOIN analytics.reservations AS r ON r.category_id = c.id
        WHERE c.hotel_id = :hotel_id
          AND r.status = 'confirmed'
        GROUP BY c.name
        ORDER BY bookings DESC
    """),
    engine,
    params={"hotel_id": selected_hotel},
)
sql_df

### Aufgabe 1b.2: Dieselbe Frage in pandas · ⏱ ca. 10 Min.

Dieselbe Abfrage könnt ihr auch mit den geladenen DataFrames beantworten. Jede SQL-Zeile hat ein pandas-Gegenstück. Zwei Dinge sind neu:

- In `query` greift ihr mit **`@`** auf eine Python-Variable zu: `query("hotel_id == @selected_hotel")`.
- **`groupby(...).size()`** zählt die Zeilen je Gruppe (wie `COUNT(*)`). **`reset_index(name="bookings")`** macht daraus wieder einen normalen DataFrame mit der Zählspalte `bookings`.

**Mini-Beispiel:**

In [ ]:
toy_sql = pd.DataFrame({
    "hotel_id": [1, 1, 1, 2],
    "category": ["Standard", "Suite", "Standard", "Standard"],
})
wanted_hotel = 1
toy_sql.query("hotel_id == @wanted_hotel").groupby("category").size().reset_index(name="bookings")

In [ ]:
# WHERE c.hotel_id = :hotel_id   (und Spalten umbenennen wie c.name AS category)
hotel_categories = categories.query("hotel_id == @selected_hotel").rename(
    columns={"id": "category_id", "name": "category"}
)
# WHERE r.status = 'confirmed'
confirmed = reservations.query("status == 'confirmed'")
# JOIN ... ON r.category_id = c.id
joined = confirmed.merge(hotel_categories, on="category_id")

# TODO: GROUP BY c.name: Nach welcher Spalte von joined wird gruppiert?
pandas_df = joined.groupby(___).size().reset_index(name="bookings")
# ORDER BY bookings DESC
pandas_df = pandas_df.sort_values("bookings", ascending=False)
pandas_df

In [ ]:
check_same_total(sql_df, pandas_df)

### Aufgabe 1b.3: Eure eigene SQL-Abfrage · ⏱ ca. 15 Min.

Das Hotel verkauft Zimmer über drei **Vertriebskanäle** (`channel`): `direct` (eigene Website), `corporate` (Firmenverträge) und `ota` (Buchungsportale). Frage: **Welcher Kanal bringt im Jahr 2024 den höchsten Durchschnittspreis pro Nacht?**

Die Abfrage ist fast fertig. Zwei Hinweise zum SQL:

- `AVG(rate_per_night)::numeric` wandelt den Mittelwert in den PostgreSQL-Typ `numeric` um, weil `ROUND(..., 2)` nur damit auf zwei Nachkommastellen rundet.
- Diesmal steht die Lücke **im SQL-Text**. Lasst ihr sie stehen, meldet PostgreSQL einen Fehler wie `column "___" does not exist`.

Erinnerung: Jede Spalte im `SELECT`, die nicht in einer Aggregatfunktion (`AVG`, `COUNT`, …) steckt, muss im `GROUP BY` stehen.

In [ ]:
# TODO: Ergänzt im SQL, wonach gruppiert wird.
channel_sql = """
    SELECT channel,
           ROUND(AVG(rate_per_night)::numeric, 2) AS avg_rate,
           COUNT(*) AS bookings
    FROM analytics.reservations
    WHERE status = 'confirmed'
      AND checkin >= DATE '2024-01-01'
      AND checkin < DATE '2025-01-01'
    GROUP BY ___
    ORDER BY avg_rate DESC
"""
channel_rates = pd.read_sql(text(channel_sql), engine)
# numeric kommt in Python als Decimal an; für Rechnungen und Plots in float umwandeln
channel_rates["avg_rate"] = channel_rates["avg_rate"].astype(float)
channel_rates

In [ ]:
check_channel_rates(channel_rates)

🤔 Warum könnte ein Kanal teurer sein als ein anderer? Denkt daran, wer über `corporate` bucht und wann gebucht wird.

### Aufgabe 1b.4: SQL ↔ pandas · ⏱ ca. 5 Min.

Zum Vergleich steht hier dieselbe Kanal-Auswertung in pandas (vorgegeben). Legt die beiden Lösungen nebeneinander: Jeder Teil der SQL-Abfrage findet sich in einer Zeile wieder.

In [ ]:
pandas_rates = (
    reservations
    .query("status == 'confirmed' and checkin >= '2024-01-01' and checkin < '2025-01-01'")  # WHERE
    .groupby("channel")                                                                   # GROUP BY
    .agg(avg_rate=("rate_per_night", "mean"), bookings=("id", "count"))                   # SELECT AVG, COUNT
    .round(2)
    .sort_values("avg_rate", ascending=False)                                             # ORDER BY
    .reset_index()
)
pandas_rates

**Merke: SQL ↔ pandas**

| SQL | pandas |
|---|---|
| `SELECT a, b` | `df[["a", "b"]]` |
| `WHERE status = 'confirmed'` | `df.query("status == 'confirmed'")` |
| `JOIN b ON a.key = b.key` | `a.merge(b, on="key")` |
| `GROUP BY x` + `COUNT(*)` | `df.groupby("x").size()` |
| `GROUP BY x` + `AVG(y) AS avg_y` | `df.groupby("x").agg(avg_y=("y", "mean"))` |
| `ORDER BY y DESC` | `df.sort_values("y", ascending=False)` |
| `CASE WHEN … THEN … ELSE … END` | `np.where(bedingung, wert_ja, wert_nein)` |

🤔 Wann lasst ihr lieber die Datenbank rechnen, wann rechnet ihr lieber in pandas?

## Block 2: Kalender-Detektive

Ein Hotel lebt vom Kalender: Feiertage, Messen, Feste und Turniere verschieben die Nachfrage. In diesem Block werdet ihr zu Detektiv*innen und prüft, ob hinter auffälligen Zahlen ein Termin steckt.

**Wichtig:** Die Termine in `events` (Tabelle `calendar_events`) sind real. Ihre Effekte auf unsere Buchungen sind synthetisch erzeugt.

Ihr arbeitet mit den DataFrames aus Block 1: `stays` (bestätigte Aufenthalte mit `month`, `year`, `checkin`, `city_id`, …), `events`, `reservations` und `guests`.

### Aufgabe 2.1: Die Osterfalle · ⏱ ca. 20 Min.

Das Management ist begeistert: Im März 2024 gab es deutlich mehr Anreisen als im März 2023. Ist das Hotel wirklich gewachsen? Oder steckt etwas anderes dahinter?

Dafür brauchen wir eine Tabelle mit **einer Zeile je Monat** und **einer Spalte je Jahr**. In jeder Zelle steht die Zahl der Anreisen. So etwas baut `pivot_table`:

- `index=` bestimmt die Zeilen,
- `columns=` bestimmt die Spalten,
- `values=` ist die Spalte, die ausgewertet wird,
- `aggfunc=` sagt, wie: `"count"` zählt, `"sum"` summiert, `"mean"` bildet den Mittelwert.

**Mini-Beispiel:** Ein Spielzeug-Hotel mit fünf Anreisen.

In [ ]:
toy = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "month": [3, 3, 4, 3, 4],
    "year": [2023, 2024, 2023, 2024, 2024],
})
toy.pivot_table(index="month", columns="year", values="id", aggfunc="count")

Lest die Tabelle: Im März (Zeile 3) 2024 gab es zwei Anreisen, im April (Zeile 4) 2023 eine.

Jetzt dasselbe mit den echten Daten. Die Zeilen sind die Monate, die Spalten die Jahre.

In [ ]:
# TODO: Welche Spalte von stays gehört in die Spalten der Tabelle?
monthly = stays.pivot_table(index="month", columns=___, values="id", aggfunc="count")
monthly

In [ ]:
check_monthly(monthly)

Wir zeichnen nur die vollständigen Jahre 2023 bis 2025. Die Liste `years` enthält nur Jahre, die es in `monthly` wirklich gibt.

In [ ]:
years = [y for y in [2023, 2024, 2025] if y in monthly.columns]
monthly[years].plot(marker="o", title="Anreisen je Monat", xlabel="Monat", ylabel="Anreisen")

Seht ihr den Sprung im März 2024? Detektiv-Frage: Welcher Feiertag wandert jedes Jahr durch den Kalender? Ostern!

Mit `query` filtert ihr Zeilen mit einer Bedingung in Anführungszeichen. Texte stehen darin in einfachen Anführungszeichen `'…'`. Mit `@` greift ihr auf eine Python-Variable zu.

**Mini-Beispiel:**

In [ ]:
toy_events = pd.DataFrame({
    "name": ["Neujahr", "Ostermontag", "Karneval", "Ostermontag"],
    "event_date": pd.to_datetime(["2023-01-01", "2023-04-10", "2023-02-16", "2024-04-01"]),
})
wanted = ["Neujahr", "Karneval"]
display(toy_events.query("name == 'Ostermontag'"))
toy_events.query("name in @wanted")

Jetzt schlagt ihr in `events` nach, auf welches Datum der Ostermontag in den einzelnen Jahren fiel.

In [ ]:
# TODO: Ergänzt den Namen des Feiertags (in einfachen Anführungszeichen).
easter = events.query("name == ___")[["event_date", "name"]]
easter

Ostern 2024 lag sehr früh: Ostersonntag war der 31. März. Viele Osterurlaube fielen also in den März statt in den April. Vergleicht März und April einzeln:

In [ ]:
monthly.loc[[3, 4], years]

Und jetzt März und April **zusammen**. `.sum()` addiert die beiden Zeilen je Jahr. Ein Ostereffekt kann so nicht mehr zwischen den Monaten hin- und herspringen.

In [ ]:
monthly.loc[[3, 4], years].sum()

🤔 Ist das Hotel im März 2024 wirklich gewachsen? Was würdet ihr dem Management sagen?

### Aufgabe 2.2: Event-Study · ⏱ ca. 30 Min.

Bringen Messen und Feste dem Hotel wirklich mehr Gäste? Ein einzelnes Jahr ist oft verrauscht. Deshalb machen wir eine **Event-Study**:

1. Wir legen alle Ausgaben eines Events übereinander, zum Beispiel das Oktoberfest 2023, 2024 und 2025.
2. **Tag 0** ist jeweils der erste Eventtag. Tag −3 heißt: drei Tage vorher, Tag 5: fünf Tage danach.
3. Für jeden Tag zählen wir die Anreisen in der Stadt des Events.
4. Zum Schluss teilen wir durch den Durchschnitt der **14 Tage davor**. Eine 1,0 heißt dann „normal“, eine 2,0 „doppelt so viele Anreisen wie sonst“.

Wir gehen in kleinen Schritten vor. Zuerst wählen wir die Events aus.

In [ ]:
names = ["Oktoberfest", "Hannover Messe", "Frankfurter Buchmesse", "gamescom", "Karneval"]
selected = events.query("name in @names").copy()
selected.head()

Ein Event steht in `events` mit **einer Zeile je Tag**. Uns interessiert aber nur der erste Tag jeder Ausgabe.

Noch ein technisches Detail: `events["city_id"]` hat den Typ `Int64` (großes I). Dieser Typ erlaubt fehlende Werte `<NA>`, z. B. für bundesweite Feiertage. In `stays` ist `city_id` ein normales `int64`. Damit `merge` später passt, entfernen wir fehlende Werte mit `dropna` und wandeln den Typ mit `astype` um.

In [ ]:
selected = selected.dropna(subset=["city_id"]).astype({"city_id": "int64"})
selected["year"] = selected["event_date"].dt.year
# Erster Tag je Event, Stadt und Jahr
starts = selected.groupby(["name", "city_id", "year"], as_index=False)["event_date"].min()
starts

Jetzt zählen wir die Anreisen je Stadt und Tag. `groupby(...).size()` zählt die Zeilen je Gruppe.

In [ ]:
city_days = stays.groupby(["city_id", "checkin"]).size().rename("bookings").reset_index()
city_days.head()

Mit `merge` hängen wir an jeden Stadt-Tag die Starttermine der Events in dieser Stadt. Eine Zeile sagt dann z. B.: „In München kamen am 10.9.2024 so viele Gäste an, und das Oktoberfest 2024 begann am 21.9.2024.“

In [ ]:
study = city_days.merge(starts, on="city_id")
study.head()

Der Kern der Event-Study ist der **Abstand in Tagen** zwischen Anreise und Eventstart. Zieht man zwei Datumswerte voneinander ab, entsteht eine Zeitdauer (`Timedelta`). `.dt.days` macht daraus eine ganze Zahl.

**Mini-Beispiel:**

In [ ]:
toy_days = pd.DataFrame({
    "checkin": pd.to_datetime(["2024-09-18", "2024-09-21", "2024-09-25"]),
    "event_date": pd.to_datetime(["2024-09-21", "2024-09-21", "2024-09-21"]),
})
toy_days["offset"] = (toy_days["checkin"] - toy_days["event_date"]).dt.days
toy_days

In [ ]:
# TODO: Von welchem Datum zieht ihr ab? Tag 0 soll der erste Eventtag sein.
study["offset"] = (study["checkin"] - study[___]).dt.days
study[["checkin", "event_date", "offset"]].head()

Wir behalten nur das Fenster von 14 Tagen vor bis 21 Tage nach dem Eventstart.

In [ ]:
study = study.query("-14 <= offset <= 21")
study["offset"].describe()

Jetzt legen wir die Jahre übereinander: Eine Zeile je `offset`, eine Spalte je Event, in der Zelle der Mittelwert der Anreisen über alle Jahre.

In [ ]:
curve = study.pivot_table(index="offset", columns="name", values="bookings", aggfunc="mean")
curve.head()

Die Events finden in unterschiedlich großen Städten statt. Damit man sie vergleichen kann, teilen wir jede Spalte durch ihren eigenen Normalwert, den Durchschnitt der Tage −14 bis −1.

`div` teilt jede Spalte durch die passende Zahl.

**Mini-Beispiel:**

In [ ]:
toy_curve = pd.DataFrame({"A": [10, 10, 30], "B": [2, 2, 3]}, index=[-2, -1, 0])
toy_base = toy_curve.loc[-2:-1].mean()
toy_curve.div(toy_base)

In [ ]:
baseline = curve.loc[-14:-1].mean()
# TODO: Teilt die Kurve durch den Normalwert der 14 Tage davor.
curve_rel = curve.div(___)
curve_rel.round(2).head()

In [ ]:
check_event_curve(curve_rel)

In [ ]:
ax = curve_rel.plot(title="Anreisen rund um Events (relativ zu den 14 Tagen davor)",
                    xlabel="Tage relativ zum Eventstart", ylabel="Anreisen (1 = normal)")
ax.axvline(0, color="grey", linestyle="--")

🤔 Bei welchen Events steigen die Anreisen am stärksten? Fällt die Kurve nach dem Event wieder auf etwa 1 zurück?

### Aufgabe 2.3: Euro 2024 · ⏱ ca. 20 Min.

Vom 14. Juni bis 14. Juli 2024 fand die Fußball-Europameisterschaft in Deutschland statt. Haben die Hotels in den Austragungsorten davon profitiert?

Ein einfacher Vergleich „Juni 2024 gegen Juni 2023“ reicht nicht, denn vielleicht lief 2024 überall besser. Deshalb vergleichen wir zwei Gruppen:

- **Austragungsort**: Städte mit EM-Spielen,
- **Vergleichsgruppe**: alle anderen Städte.

Beide jeweils im Turnierzeitraum vom 14. Juni bis 14. Juli, relativ zu 2023. Warum nicht einfach die ganzen Monate Juni und Juli? Dann würden die Tage ohne Turnier den Effekt verwässern.

Für die Einteilung braucht ihr zwei Werkzeuge:

- `isin(liste)` prüft für jede Zeile, ob der Wert in der Liste vorkommt, und gibt `True`/`False` zurück.
- `np.where(bedingung, wert_wenn_true, wert_wenn_false)` macht daraus Text.

**Mini-Beispiel:**

In [ ]:
toy_stays = pd.DataFrame({"city_id": [1, 2, 3, 2]})
toy_hosts = [2, 3]
toy_stays["is_host"] = toy_stays["city_id"].isin(toy_hosts)
toy_stays["group"] = np.where(toy_stays["is_host"], "Austragungsort", "Vergleichsgruppe")
toy_stays

Zuerst holen wir die Städte mit EM-Spielen aus `events`. Wieder entfernen wir fehlende Werte und wandeln in `int64` um.

Für den Zeitraum nutzen wir einen kleinen Trick: `.dt.strftime("%m-%d")` macht aus jedem Datum Text wie `"06-14"` (Monat-Tag, ohne Jahr). Mit `.between("06-14", "07-14")` behalten wir dann in **jedem** Jahr genau den Turnierzeitraum.

In [ ]:
hosts = events.query("name == 'Euro 2024'")["city_id"].dropna().astype("int64").unique()
in_window = stays["checkin"].dt.strftime("%m-%d").between("06-14", "07-14")
summer = stays[in_window].query("year in [2023, 2024, 2025]").copy()
hosts

In [ ]:
# TODO: Welche Liste enthält die Austragungsorte?
summer["group"] = np.where(summer["city_id"].isin(___), "Austragungsort", "Vergleichsgruppe")
summer["group"].value_counts()

Jetzt zählen wir die Anreisen je Jahr und Gruppe und teilen durch das Jahr 2023. 2023 steht dann bei beiden Gruppen auf 1,0.

In [ ]:
euro = summer.pivot_table(index="year", columns="group", values="id", aggfunc="count")
euro_rel = euro.div(euro.loc[2023])
euro_rel.round(2)

In [ ]:
check_euro(euro_rel)

In [ ]:
euro_rel.plot.bar(title="Anreisen 14.6.–14.7. relativ zu 2023", xlabel="Jahr", ylabel="Anreisen (2023 = 1)")

🤔 Was passiert 2024 in den Austragungsorten im Vergleich zur Vergleichsgruppe? Und warum ist die Vergleichsgruppe wichtig?

### Aufgabe 2.4: Vertiefung: Stornos und Familien · ⏱ ca. 15 Min.

Diese Aufgabe ist freiwillig, falls ihr noch Zeit habt.

**Wer storniert am häufigsten?** Dafür brauchen wir alle Reservierungen, auch die stornierten, also `reservations` statt `stays`.

Ein Trick: Der Mittelwert einer `True`/`False`-Spalte ist der **Anteil** der `True`-Werte, denn `True` zählt als 1 und `False` als 0.

**Mini-Beispiel:**

In [ ]:
toy_res = pd.DataFrame({
    "channel": ["direct", "direct", "ota", "ota"],
    "status": ["confirmed", "cancelled", "cancelled", "cancelled"],
})
toy_cancelled = toy_res["status"] == "cancelled"
toy_cancelled.groupby(toy_res["channel"]).mean()

In [ ]:
# TODO: Welcher Status steht für eine Stornierung?
cancelled = reservations["status"] == ___
cancel_rate = cancelled.groupby(reservations["channel"]).mean()
cancel_rate.round(3)

In [ ]:
check_cancel_rate(cancel_rate)

In [ ]:
cancel_rate.plot.bar(title="Stornoquote je Buchungskanal", xlabel="Kanal", ylabel="Anteil storniert")

**Wann reisen Familien aus welchem Bundesland?** Wir nehmen Aufenthalte mit Kindern im Sommer 2025 und holen mit `merge` das Herkunfts-Bundesland `home_state` aus `guests`. `dt.isocalendar().week` liefert die Kalenderwoche. Hier gibt es nichts zu ergänzen, lest den Code und schaut euch den Plot an.

In [ ]:
families = stays.query("children > 0 and year == 2025 and month in [6, 7, 8, 9]")
families = families.merge(guests[["id", "home_state"]], left_on="guest_id", right_on="id", suffixes=("", "_guest"))
families["week"] = families["checkin"].dt.isocalendar().week
family_weeks = families.pivot_table(index="week", columns="home_state", values="id", aggfunc="count")
family_weeks.plot(title="Anreisen von Familien je Kalenderwoche (Sommer 2025)",
                  xlabel="Kalenderwoche", ylabel="Anreisen")

🤔 Warum haben die Bundesländer ihre Gipfel in unterschiedlichen Wochen? Was bedeutet das für ein Hotel, das Familien anspricht?

## Block 3: Preisfalle und Umsatzkurve

Die Hotelleitung stellt eine einfache Frage: **Welcher Preis bringt uns den höchsten Umsatz?**

Klingt nach einer Zeile Code. In diesem Block seht ihr, warum die naheliegende Antwort
falsch ist und wie man mit den richtigen Daten trotzdem eine faire Antwort findet.
Wir gehen in kleinen Schritten vor:

1. Eine Tabelle bauen: ein Angebotspreis und die Zahl der Buchungen je Kategorie und Anreisetag.
2. Einen ersten Blick auf Preis und Verkäufe werfen.
3. Verstehen, woher die überraschende Kurve kommt.
4. Eine faire Preiskurve und den „Umsatzberg“ finden.
5. Prüfen, ob das für alle Gäste und alle Vertriebskanäle gilt.

Ihr braucht dafür `stays`, `places`, `daily_rates` und `events` aus Block 1.

**Wichtige Kennzahl `r`:** Jede Zimmerkategorie hat einen Listenpreis (`base_price`).
Das Hotel bietet an jedem Tag einen Preis an (`offered_rate`). Wir rechnen
`r = offered_rate / base_price`. `r = 1.0` heißt Listenpreis, `r = 0.8` heißt 20 % günstiger,
`r = 1.2` heißt 20 % teurer. So lassen sich Kategorien mit verschiedenen Listenpreisen vergleichen.

### Aufgabe 3.1: Eine Tabelle je Kategorie und Anreisetag · ⏱ ca. 15 Min.

In `daily_rates` steht für jede Kategorie und jeden Anreisetag der angebotene Preis.
In `stays` stehen die tatsächlichen Buchungen. Wir wollen beides in **einer** Tabelle `daily`.

Zuerst zählen wir die Buchungen je Kategorie und Anreisetag. Dieser Code ist vorgegeben.

In [ ]:
# Buchungen je Kategorie und Anreisetag zählen
sales = stays.groupby(["category_id", "checkin"]).size()  # size() zählt die Zeilen je Gruppe
sales = sales.rename("bookings").reset_index()            # Series -> DataFrame mit Spalte bookings
sales = sales.rename(columns={"checkin": "arrival_date"}) # gleicher Spaltenname wie in daily_rates
sales.head()

**Das Problem:** In `sales` fehlen alle Tage, an denen **niemand** gebucht hat.
Gerade diese Tage sind für die Preisfrage wichtig: Vielleicht war der Preis dort zu hoch.

**Mini-Beispiel `merge` mit `how`:** Beim Verbinden zweier Tabellen entscheidet `how`,
welche Zeilen übrig bleiben.

- `how="inner"`: nur Zeilen, die es in **beiden** Tabellen gibt.
- `how="left"`: **alle** Zeilen der linken Tabelle. Fehlt rechts ein Partner, steht dort `NaN` (leer).

In [ ]:
demo_days = pd.DataFrame({"day": ["Mo", "Di", "Mi"], "price": [80, 90, 100]})
demo_sales = pd.DataFrame({"day": ["Mo", "Mi"], "bookings": [3, 1]})

print("how='inner':")
print(demo_days.merge(demo_sales, how="inner", on="day"))
print()
print("how='left':")
print(demo_days.merge(demo_sales, how="left", on="day"))

Beim `inner`-Merge ist der Dienstag verschwunden. Beim `left`-Merge bleibt er, aber mit `NaN`.
Mit `fillna(wert)` ersetzt ihr leere Werte. Ein Tag ohne Buchung hat 0 Buchungen.

**Mini-Beispiel `fillna`:**

In [ ]:
demo_left = demo_days.merge(demo_sales, how="left", on="day")
demo_left["bookings"] = demo_left["bookings"].fillna(0)
demo_left

Jetzt ihr. `daily` soll **jeden** Tag aus `daily_rates` behalten, auch Tage ohne Verkauf.

In [ ]:
# Schritt 1: Preise mit den Kategorie-Infos (Listenpreis, Stadt, ...) verbinden
daily = daily_rates.merge(places, on="category_id")

# TODO: Verbindet daily mit sales so, dass alle Tage erhalten bleiben,
#       und tragt für Tage ohne Verkauf die richtige Zahl ein.
daily = daily.merge(sales, how=___, on=["category_id", "arrival_date"])
daily["bookings"] = daily["bookings"].fillna(___)

# Preis im Verhältnis zum Listenpreis, auf 2 Nachkommastellen gerundet
daily["r"] = (daily["offered_rate"] / daily["base_price"]).round(2)
daily[["category", "arrival_date", "offered_rate", "base_price", "r", "bookings"]].head()

In [ ]:
check_daily(daily, daily_rates)

### Aufgabe 3.2: Der erste Blick · ⏱ ca. 10 Min.

Im normalen Betrieb setzt das Hotel seine Preise selbst (`price_source == 'dynamic'`).
So sieht die Hotelleitung ihre Zahlen jeden Tag. Wir zeichnen für jeden Wert von `r`,
wie viele Buchungen ein Tag im Durchschnitt hatte. Der Code ist vorgegeben.

In [ ]:
regular_days = daily.query("price_source == 'dynamic'")

regular_days.groupby("r")["bookings"].mean().plot(
    marker="o",
    title="Regelbetrieb: Preis gegen Verkäufe",
    xlabel="r (Angebotspreis / Listenpreis)",
    ylabel="Buchungen je Tag",
)
plt.show()

🤔 Was seht ihr? Was würdet ihr der Hotelleitung raten, wenn ihr nur dieses Bild hättet?

### Aufgabe 3.3: Woher kommt diese Kurve? · ⏱ ca. 20 Min.

Glaubt ihr wirklich, dass höhere Preise mehr Gäste anlocken? Sucht eine andere Erklärung.
Ein Verdacht: An manchen Tagen wollen ohnehin viele Menschen in die Stadt, zum Beispiel
bei einer Messe, einem Festival oder im Sommer. Wir markieren deshalb jeden Tag mit einer **Tagesart**.

**Schritt 1: Event-Tage markieren.** In `events` stehen Termine je Stadt.
Bundesweite Termine (`city_id` leer) lassen wir hier weg.

**Mini-Beispiel „Markierung per Merge“:** Wir geben der Event-Tabelle eine Spalte `is_event = True`
und verbinden mit `how="left"`. Wo es kein Event gibt, steht danach `NaN`.
`.notna()` macht daraus `True`/`False`.

In [ ]:
demo_days = pd.DataFrame({"day": ["Mo", "Di", "Mi"], "price": [80, 120, 90]})
demo_events = pd.DataFrame({"day": ["Di"]}).assign(is_event=True)

demo_marked = demo_days.merge(demo_events, how="left", on="day")
print(demo_marked)
print()
demo_marked["is_event"] = demo_marked["is_event"].notna()
print(demo_marked)

Jetzt für unsere Daten. Achtung, Datentypen: In `events` ist `city_id` vom Typ `Int64`
(erlaubt leere Werte), in `daily` vom Typ `int64`. Nach `dropna` wandeln wir mit `astype("int64")` um,
damit der Merge sauber passt.

In [ ]:
# Nur Events mit Stadt, eine Zeile je Stadt und Tag
city_events = events.dropna(subset=["city_id"])[["city_id", "event_date"]].drop_duplicates()
city_events["city_id"] = city_events["city_id"].astype("int64")
city_events = city_events.rename(columns={"event_date": "arrival_date"}).assign(is_event=True)

# TODO: Verbindet so, dass alle Tage aus regular_days erhalten bleiben.
regular = regular_days.merge(city_events, how=___, on=["city_id", "arrival_date"])
regular["is_event"] = regular["is_event"].notna()
regular["is_event"].value_counts()

**Schritt 2: Tagesart festlegen.** Jeder Tag bekommt genau eine von drei Tagesarten:

1. `"Event"`, wenn ein Event stattfindet,
2. sonst `"Sommer"`, wenn der Tag im Juni, Juli oder August liegt,
3. sonst `"Alltag"`.

**Mini-Beispiel verschachteltes `np.where`:** `np.where(bedingung, wert_wenn_ja, wert_wenn_nein)`.
Für drei Fälle setzt ihr ein zweites `np.where` in den Nein-Teil. Mit `.isin([...])` prüft ihr,
ob ein Wert in einer Liste vorkommt.

In [ ]:
demo = pd.DataFrame({
    "day": ["10.03.", "15.07.", "20.07.", "05.11."],
    "month": [3, 7, 7, 11],
    "is_event": [False, False, True, True],
})
demo_summer = demo["month"].isin([6, 7, 8])
demo["day_type"] = np.where(demo["is_event"], "Event", np.where(demo_summer, "Sommer", "Alltag"))
demo

Beachtet die Reihenfolge: Der 20.07. ist Sommer **und** Event, er wird zu `"Event"`,
weil die Event-Bedingung zuerst geprüft wird.

In [ ]:
# TODO: Legt fest, welche Monate Sommer sind, und setzt die Sommer-Bedingung ins innere np.where.
is_summer = regular["arrival_date"].dt.month.isin(___)
regular["day_type"] = np.where(regular["is_event"], "Event", np.where(___, "Sommer", "Alltag"))
regular["day_type"].value_counts()

In [ ]:
check_day_type(regular)

**Schritt 3: Vergleichen.** Wie hoch sind Preis `r` und Buchungen im Durchschnitt je Tagesart?
Der Code ist vorgegeben, lest das Ergebnis genau.

In [ ]:
regular.groupby("day_type")[["r", "bookings"]].mean().round(2)

**Schritt 4: Die Kurve getrennt nach Tagesart.**

**Mini-Beispiel `pivot_table`:** `pivot_table` macht aus einer langen Tabelle eine Kreuztabelle.
`index` wird zu den Zeilen, `columns` zu den Spalten, und in den Zellen steht
`aggfunc` (hier der Mittelwert) über `values`.

In [ ]:
demo = pd.DataFrame({
    "r": [0.9, 0.9, 1.1, 1.1, 1.1],
    "day_type": ["Alltag", "Event", "Alltag", "Event", "Event"],
    "bookings": [2, 5, 1, 6, 4],
})
demo.pivot_table(index="r", columns="day_type", values="bookings", aggfunc="mean")

Jetzt für unsere Daten. Jede Tagesart bekommt eine eigene Farbe. Der Code ist vorgegeben.

In [ ]:
points = regular.pivot_table(index="r", columns="day_type", values="bookings", aggfunc="mean")
points.plot(
    style="o",
    alpha=0.6,
    title="Regelbetrieb: Preis gegen Verkäufe nach Tagesart",
    xlabel="r (Angebotspreis / Listenpreis)",
    ylabel="Buchungen je Tag",
)
plt.show()

🤔 Wo liegen die Event-Tage, wo die Alltagstage? Was ist hier Ursache, was ist Wirkung?

### Aufgabe 3.4: Verkäufe sind nicht Nachfrage · ⏱ ca. 10 Min.

Es gibt ein zweites Problem. An manchen Tagen ist eine Kategorie ausgebucht und das Hotel
nimmt keine neuen Anreisen mehr an. Das steht in der Spalte `closed_to_arrival` (`True`/`False`).

**Mini-Beispiel: Anteil mit `mean`.** Bei `True`/`False` zählt pandas `True` als 1 und `False` als 0.
Der Mittelwert ist deshalb der **Anteil** der `True`-Werte.

In [ ]:
demo_closed = pd.Series([True, False, False, True])
print("Anzahl True:", demo_closed.sum())
print("Anteil True:", demo_closed.mean())

In [ ]:
# TODO: Berechnet den Anteil der Tage, an denen keine Anreise mehr möglich war.
share_closed = daily["closed_to_arrival"].___()
print(f"Anteil geschlossener Tage: {share_closed:.1%}")

In [ ]:
check_closed_share(share_closed)

Wie viele Buchungen gibt es an geschlossenen und offenen Tagen, je Zimmerkategorie?
Nutzt wieder `pivot_table`: Zeilen sind die Kategorien, Spalten sind offen/geschlossen.

In [ ]:
# TODO: Setzt die Spalte ein, nach der ihr offene und geschlossene Tage trennen wollt.
closed_table = daily.pivot_table(index="category", columns=___, values="bookings", aggfunc="mean")
closed_table.round(1)

🤔 Sagt die Zahl der Buchungen an geschlossenen Tagen, wie viele Gäste gern gekommen wären?

### Aufgabe 3.5: Eine faire Preiskurve · ⏱ ca. 15 Min.

Für eine faire Aussage brauchen wir Tage, an denen der Preis **nicht** von der Nachfrage abhängt.
Schaut euch dazu noch einmal die Spalten von `daily` und die Werte von `price_source` an.

In [ ]:
print(daily.columns.tolist())
daily["price_source"].value_counts()

Neben `'dynamic'` gibt es `'test'`. An diesen Tagen hat das Hotel den Preis **zufällig** gewählt,
zum Beispiel per Losverfahren, ganz egal ob Messe, Sommer oder Alltag. Das ist ein **Experiment**,
wie ein A/B-Test bei einem Online-Shop: Weil der Zufall entscheidet, sind teure und günstige Testtage
im Mittel gleich „gute“ Tage. Unterschiede bei den Verkäufen kommen dann vom Preis selbst.

**Mini-Beispiel `query`:** Die Bedingung steht als Text in Anführungszeichen. Ein Textwert
darin bekommt **einfache** Anführungszeichen.

In [ ]:
demo = pd.DataFrame({"fruit": ["Apfel", "Birne", "Apfel"], "price": [1.0, 1.5, 1.2]})
demo.query("fruit == 'Apfel'")

In [ ]:
# TODO: Behaltet nur die Tage mit zufälligem Preistest.
#       Der Wert gehört in einfache Anführungszeichen innerhalb des Textes.
test = daily.query("price_source == ___")
len(test)

In [ ]:
check_test_days(test)

Jetzt die Kurve: mittlere Buchungen je Testtag für jeden Wert von `r`.

**Mini-Beispiel `agg` mit Namen:** `agg(neuer_name=("spalte", "funktion"))` rechnet je Gruppe
eine Kennzahl und gibt der Ergebnisspalte gleich einen sprechenden Namen.

In [ ]:
demo = pd.DataFrame({"r": [0.8, 0.8, 1.0, 1.0], "bookings": [4, 6, 2, 3]})
demo.groupby("r").agg(rooms=("bookings", "mean"))

In [ ]:
# TODO: Berechnet je Preisstufe r die mittlere Zahl der Buchungen.
curve = test.groupby("r").agg(rooms=("bookings", ___))
curve.head()

In [ ]:
curve["rooms"].plot(
    marker="o",
    title="Preistest: Preis gegen Verkäufe",
    xlabel="r (Angebotspreis / Listenpreis)",
    ylabel="Buchungen je Testtag",
)
plt.show()

🤔 Vergleicht mit dem Bild aus Aufgabe 3.2. Was ist anders?

### Aufgabe 3.6: Der Umsatzberg · ⏱ ca. 10 Min.

Ein niedriger Preis bringt viele Buchungen, aber wenig Geld je Zimmer. Ein hoher Preis bringt viel
Geld je Zimmer, aber wenige Buchungen. Umsatz ist **Preis mal Menge**.
Wir rechnen in Vielfachen des Listenpreises: Umsatz je Testtag = `r` × `rooms`.
Die Preisstufe `r` steht im Index von `curve` (`curve.index`).

**Mini-Beispiel `max` und `idxmax`:** `max()` liefert den größten **Wert**,
`idxmax()` liefert die **Stelle** (den Index), an der er steht.

In [ ]:
demo = pd.Series([3.0, 7.0, 5.0], index=[0.8, 0.9, 1.0])
print("größter Wert:   ", demo.max())
print("an der Stelle r:", demo.idxmax())

In [ ]:
# TODO: Umsatz = Preis mal Menge. Der Preis steckt im Index.
curve["revenue"] = ___ * curve["rooms"]
curve["revenue"].plot(
    marker="o",
    title="Preistest: Umsatz je Preisstufe",
    xlabel="r (Angebotspreis / Listenpreis)",
    ylabel="Umsatz je Testtag (in Listenpreisen)",
)
plt.show()

In [ ]:
# TODO: Bei welchem r liegt der Gipfel des Umsatzbergs?
best_r = curve["revenue"].___()
best_r

In [ ]:
check_revenue_peak(best_r)

🤔 Liegt der beste Preis über oder unter dem Listenpreis? Was hätte die Hotelleitung
mit dem Bild aus Aufgabe 3.2 entschieden?

### Aufgabe 3.7: Gilt das für alle Gäste? · ⏱ ca. 20 Min.

In Block 1 habt ihr die Gäste in `business` und `leisure` eingeteilt. Reagieren beide Gruppen
gleich auf den Preis? Dafür brauchen wir die einzelnen Aufenthalte auf Testtagen.

Der erste Schritt ist vorgegeben: Wir verbinden jeden Aufenthalt aus `stays` mit seinem Testtag
(gleiche Kategorie, Anreise = Testtag) und holen uns so das `r`. Ein normaler (`inner`) Merge
genügt, denn Aufenthalte an Nicht-Testtagen wollen wir hier gar nicht.

In [ ]:
if "segment" not in stays.columns:
    print("💡 In stays fehlt noch die Spalte segment. Löst zuerst die Segment-Aufgabe in Block 1,")
    print("   führt die Zelle dort aus und kommt dann hierher zurück.")
else:
    columns_needed = ["category_id", "arrival_date", "r"]
    test_stays = stays.merge(
        test[columns_needed],
        left_on=["category_id", "checkin"],
        right_on=["category_id", "arrival_date"],
    )
    # Wie viele Testtage gab es je Preisstufe? Damit rechnen wir gleich "je Testtag".
    days_per_r = test.groupby("r").size()
    print("Aufenthalte auf Testtagen:", len(test_stays))
    print(test_stays["segment"].value_counts())

Für manche Preisstufen gibt es mehr Testtage als für andere. Damit der Vergleich fair ist,
teilen wir die Zahl der Aufenthalte durch die Zahl der Testtage je `r`.

**Mini-Beispiel `pivot_table` mit `count` und `div(..., axis=0)`:** `aggfunc="count"` zählt Zeilen.
`div(series, axis=0)` teilt jede **Zeile** der Tabelle durch den passenden Wert der Series
(gleicher Index).

In [ ]:
demo = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6],
    "r": [0.8, 0.8, 0.8, 1.2, 1.2, 1.2],
    "segment": ["leisure", "leisure", "business", "business", "business", "leisure"],
})
demo_days_per_r = pd.Series([2, 1], index=[0.8, 1.2])

demo_counts = demo.pivot_table(index="r", columns="segment", values="id", aggfunc="count")
print(demo_counts)
print()
print(demo_counts.div(demo_days_per_r, axis=0))

In [ ]:
# TODO: Zählt die Aufenthalte je Preisstufe r, getrennt nach Gästegruppe.
rooms_by_segment = test_stays.pivot_table(index="r", columns=___, values="id", aggfunc="count")
rooms_by_segment = rooms_by_segment.div(days_per_r, axis=0)  # Aufenthalte je Testtag
rooms_by_segment.round(2).head()

Jetzt wieder Preis mal Menge, diesmal für beide Gruppen gleichzeitig.
`mul(series, axis=0)` multipliziert jede Zeile mit dem passenden Wert, genau wie `div` oben teilt.

In [ ]:
# TODO: Multipliziert jede Zeile mit ihrer Preisstufe r (steht im Index).
revenue_by_segment = rooms_by_segment.mul(___, axis=0)
revenue_by_segment.plot(
    marker="o",
    title="Preistest: Umsatz je Gästegruppe",
    xlabel="r (Angebotspreis / Listenpreis)",
    ylabel="Umsatz je Testtag (in Listenpreisen)",
)
plt.show()

In [ ]:
# TODO: Bei welchem r liegt der Gipfel für jede Gruppe? (idxmax arbeitet auf jeder Spalte einzeln)
peaks = revenue_by_segment.___()
peaks

In [ ]:
check_segment_peaks(peaks)

🤔 Warum reagieren die beiden Gruppen so unterschiedlich auf den Preis?
Was bedeutet das für die Preisgestaltung des Hotels?

### Aufgabe 3.8: Brutto oder netto? · ⏱ ca. 15 Min.

Bisher haben wir den Umsatz gezählt, den Gäste bezahlen (**brutto**). Buchungsportale verlangen aber
eine Provision. Dem Hotel bleibt nur der **Nettobetrag**. In `test_stays` stehen dafür
`rate_per_night` (Preis je Nacht), `nights` (Zahl der Nächte), `channel` (Vertriebskanal) und
`commission_pct` (Provision in Prozent).

**Mini-Beispiel brutto und netto:** Bei 20 % Provision bleiben 80 %, also `1 - 20 / 100 = 0.8`.

In [ ]:
demo = pd.DataFrame({
    "rate_per_night": [100.0, 100.0, 120.0],
    "nights": [2, 2, 1],
    "commission_pct": [0.0, 20.0, 10.0],
})
demo["gross"] = demo["rate_per_night"] * demo["nights"]
demo["net"] = demo["gross"] * (1 - demo["commission_pct"] / 100)
demo

In [ ]:
test_stays["gross"] = test_stays["rate_per_night"] * test_stays["nights"]

# TODO: Zieht die Provision ab. commission_pct ist in Prozent angegeben.
test_stays["net"] = test_stays["gross"] * (1 - ___ / 100)
test_stays[["channel", "commission_pct", "gross", "net"]].head()

In [ ]:
check_net(test_stays)

Jetzt die Umsatzkurve in Euro je Testtag, brutto und netto. Der Code ist vorgegeben.

In [ ]:
money = test_stays.groupby("r")[["gross", "net"]].sum().div(days_per_r, axis=0)
money.plot(
    marker="o",
    title="Preistest: Umsatz je Testtag, brutto und netto",
    xlabel="r (Angebotspreis / Listenpreis)",
    ylabel="Euro je Testtag",
)
plt.show()

Und je Vertriebskanal: Wie viel vom Bruttoumsatz bleibt dem Hotel? Der Code ist vorgegeben.

In [ ]:
channel_money = test_stays.groupby("channel")[["gross", "net"]].sum()
channel_money["kept_pct"] = 100 * channel_money["net"] / channel_money["gross"]
channel_money.round(1)

🤔 Welcher Kanal kostet das Hotel am meisten? Warum lohnt es sich für ein Hotel,
Gäste zur Direktbuchung zu bewegen?

## Block 4: SpiegelMining im Hotel (Vertiefung)

David Kriesel hat in seinem Vortrag „SpiegelMining“ aus öffentlich zugänglichen Artikeldaten erstaunlich viel
herausgelesen: Wer schreibt mit wem, wer ist wann im Urlaub, wer verschwindet plötzlich aus der Redaktion?
Geht so etwas auch mit Buchungsdaten?

Eure Leitfrage für diesen Block: **Fällt euch in den Aufenthalten etwas auf, das kein Zufall sein kann?**

Die Idee ist einfach: Wenn zwei Gäste auffällig oft **am selben Tag im selben Hotel** anreisen, kennen sie sich
vermutlich. Wir gehen in kleinen Schritten vor:

1. Gäste mit genug Aufenthalten auswählen
2. Alle Paare bilden, die am selben Tag im selben Hotel anreisen
3. Paare zählen
4. und 5. Harmlose Erklärungen aussortieren
6. Namen dazuholen
7. Die Definition von „zusammen“ verändern

Ihr braucht dafür nur `stays`, `guests` und `events` aus Block 1.

### Aufgabe 4.1: Stammgäste auswählen · ⏱ ca. 10 Min.

Wer nur ein- oder zweimal im Hotel war, kann kaum ein auffälliges Muster zeigen. Außerdem wird der Merge im
nächsten Schritt sehr groß, wenn wir alle Gäste mitnehmen. Deshalb behalten wir nur Gäste mit **mindestens
4 Aufenthalten**.

Dafür brauchen wir für **jede Zeile** die Anzahl der Aufenthalte ihres Gastes. Genau das macht
`groupby(...).transform("size")`:

- `agg("size")` fasst zusammen: **eine Zeile pro Gruppe**.
- `transform("size")` rechnet dasselbe, schreibt das Ergebnis aber **in jede Zeile der Gruppe zurück**. Das
  Ergebnis ist genauso lang wie die Tabelle und passt deshalb direkt als Filter.

In [ ]:
# Mini-Beispiel: Gast 1 war dreimal da, Gast 2 einmal
toy_stays = pd.DataFrame({
    "guest_id": [1, 1, 1, 2],
    "checkin": pd.to_datetime(["2024-01-08", "2024-02-12", "2024-03-04", "2024-01-08"]),
})
print("agg: eine Zeile pro Gast")
print(toy_stays.groupby("guest_id")["checkin"].agg("size"))
print()
print("transform: eine Zeile pro Aufenthalt")
print(toy_stays.groupby("guest_id")["checkin"].transform("size"))

In [ ]:
# TODO: Behaltet nur Gäste mit mindestens 4 Aufenthalten.
guest_stays = stays[["guest_id", "hotel_id", "city_id", "checkin"]]
stay_count = guest_stays.groupby("guest_id")["checkin"].transform("size")
guest_stays = guest_stays[stay_count >= ___]
print(f"{len(guest_stays):,} Aufenthalte von {guest_stays['guest_id'].nunique():,} Gästen")

In [ ]:
check_guest_stays(guest_stays)

### Aufgabe 4.2: Wer reist am selben Tag an? · ⏱ ca. 20 Min.

Jetzt suchen wir alle Paare von Gästen, die **im selben Hotel am selben Tag** anreisen. Der Trick:
Wir mergen die Tabelle **mit sich selbst**. Damit wir die beiden Seiten auseinanderhalten können, benennen wir
`guest_id` links in `a` und rechts in `b` um.

Dabei entsteht zu viel:

- **Selbstpaare:** Jeder Gast passt zu sich selbst, z. B. (1, 1).
- **Doppelte Paare:** Das Paar (1, 2) taucht auch als (2, 1) auf.

Beides entfernt `query("a < b")` in einem Schritt: (1, 1) fällt raus, weil 1 nicht kleiner als 1 ist, und von
(1, 2) und (2, 1) bleibt nur (1, 2). Schaut es euch im Mini-Beispiel an.

In [ ]:
# Mini-Beispiel: Gäste 1, 2 und 3 reisen am selben Tag in Hotel 10 an, Gast 4 in Hotel 20
toy_stays = pd.DataFrame({
    "guest_id": [1, 2, 3, 4],
    "hotel_id": [10, 10, 10, 20],
    "checkin": pd.to_datetime(["2024-05-06"] * 4),
})
toy_left = toy_stays.rename(columns={"guest_id": "a"})
toy_right = toy_stays.rename(columns={"guest_id": "b"})
toy_merged = toy_left.merge(toy_right, on=["hotel_id", "checkin"])
print(f"Nach dem Merge: {len(toy_merged)} Zeilen (mit Selbstpaaren und doppelten Paaren)")
print(toy_merged)
print()
print("Nach query('a < b'): jedes Paar genau einmal")
print(toy_merged.query("a < b"))

Jetzt mit den echten Daten. Die linke Seite behält `city_id`, die brauchen wir später noch.

In [ ]:
# TODO: Merged auf die beiden Spalten, die "selbes Hotel, selber Tag" beschreiben.
left = guest_stays.rename(columns={"guest_id": "a"})
right = guest_stays.rename(columns={"guest_id": "b"})[["b", "hotel_id", "checkin"]]
pairs = left.merge(right, on=[___, ___]).query("a < b")
print(f"{len(pairs):,} gemeinsame Anreisen")
pairs.head()

In [ ]:
check_pairs(pairs)

### Aufgabe 4.3: Paare zählen · ⏱ ca. 10 Min.

Jede Zeile in `pairs` ist **eine** gemeinsame Anreise. Wie oft hat sich jedes Paar getroffen? Dafür gruppieren
wir nach `a` und `b`, zählen mit `size()` und sortieren mit `sort_values(ascending=False)` absteigend.

In [ ]:
# Mini-Beispiel: Paar (1, 2) trifft sich zweimal, Paar (3, 4) einmal
toy_pairs = pd.DataFrame({"a": [1, 1, 3], "b": [2, 2, 4], "hotel_id": [10, 20, 10]})
toy_pairs.groupby(["a", "b"]).size().sort_values(ascending=False)

In [ ]:
# TODO: Sortiert so, dass die häufigsten Paare oben stehen.
top_pairs = pairs.groupby(["a", "b"]).size().sort_values(ascending=___).head(10)
top_pairs

In [ ]:
check_top_pairs(top_pairs)

Schaut euch das Spitzenpaar genauer an: In welchen Hotels treffen sich die beiden?

In [ ]:
top_a, top_b = top_pairs.index[0]
top_meetings = pairs[(pairs["a"] == top_a) & (pairs["b"] == top_b)]
print(f"Paar {top_a} und {top_b}: {len(top_meetings)} gemeinsame Anreisen")
top_meetings["hotel_id"].value_counts()

🤔 Ist das Spitzenpaar wirklich verdächtig? Welche harmlose Erklärung fällt euch ein, wenn sich zwei Menschen
sehr oft, aber immer im selben Hotel treffen?

### Aufgabe 4.4: Filter 1 – nur Paare, die herumkommen · ⏱ ca. 15 Min.

Berater treffen sich immer im selben Hotel. Ein Paar, das sich in **vielen verschiedenen Hotels** trifft, ist
viel interessanter. Wir berechnen deshalb pro Paar zwei Kennzahlen auf einmal:

- `meetings`: Anzahl gemeinsamer Anreisen (`"size"`)
- `hotels`: Anzahl **verschiedener** Hotels (`"nunique"`)

Das geht mit **benannter Aggregation**: `agg(neuer_name=("spalte", "funktion"), ...)`.

In [ ]:
# Mini-Beispiel: Paar (1, 2) trifft sich dreimal in zwei Hotels, Paar (3, 4) zweimal im selben Hotel
toy_pairs = pd.DataFrame({
    "a": [1, 1, 1, 3, 3],
    "b": [2, 2, 2, 4, 4],
    "hotel_id": [10, 10, 20, 30, 30],
})
toy_pairs.groupby(["a", "b"]).agg(meetings=("hotel_id", "size"), hotels=("hotel_id", "nunique"))

In [ ]:
# TODO: Zählt die verschiedenen Hotels pro Paar und behaltet Paare mit mindestens 4 Hotels.
pair_stats = pairs.groupby(["a", "b"]).agg(meetings=("hotel_id", "size"), hotels=("hotel_id", ___))
candidates = pair_stats[pair_stats["hotels"] >= ___].reset_index()[["a", "b"]]
print(f"{len(candidates):,} Paare treffen sich in mindestens 4 Hotels")
candidates.head()

In [ ]:
check_candidates(candidates, pairs)

### Aufgabe 4.5: Filter 2 – Messetage herausnehmen · ⏱ ca. 20 Min.

Noch eine harmlose Erklärung: Auf einer **Messe** reisen tausende Menschen am selben Tag in dieselbe Stadt.
Zwei Messebesucher treffen sich also zufällig, auch in vielen verschiedenen Hotels. Deshalb werfen wir alle
gemeinsamen Anreisen an Messetagen raus.

Wir wollen also die Zeilen behalten, die **keinen** Partner in der Messetabelle haben. Das nennt man
**Anti-Join**. In pandas geht er in zwei Schritten:

1. `merge(..., how="left", indicator=True)` behält alle linken Zeilen und schreibt in die neue Spalte `_merge`,
   ob es einen Partner gab (`both`) oder nicht (`left_only`).
2. `query("_merge == 'left_only'")` behält nur die Zeilen ohne Partner.

In [ ]:
# Mini-Beispiel: In Stadt 1 ist am 6. Mai Messe
toy_meetings = pd.DataFrame({
    "city_id": [1, 1, 2],
    "checkin": pd.to_datetime(["2024-05-06", "2024-05-13", "2024-05-06"]),
})
toy_fairs = pd.DataFrame({"city_id": [1], "checkin": pd.to_datetime(["2024-05-06"])})
toy_marked = toy_meetings.merge(toy_fairs, how="left", on=["city_id", "checkin"], indicator=True)
print(toy_marked)
print()
print("Anti-Join: nur Zeilen ohne Messe")
print(toy_marked.query("_merge == 'left_only'"))

Zuerst bauen wir die Messetabelle. Sie ist vorgegeben: In `events` heißt die Datumsspalte `event_date`, wir
benennen sie in `checkin` um, damit der Merge passt. Bundesweite Termine haben keine `city_id`; die werfen wir
mit `dropna()` raus und machen `city_id` dann zu einer normalen Ganzzahl-Spalte.

In [ ]:
fairs = events.query("event_type == 'trade_fair'")[["city_id", "event_date"]]
fairs = fairs.dropna().drop_duplicates()
fairs = fairs.rename(columns={"event_date": "checkin"}).astype({"city_id": "int64"})
print(f"{len(fairs):,} Messetage")
fairs.head()

Jetzt der Anti-Join. Wir nehmen nur die Treffen der Kandidaten aus Aufgabe 4.4 und werfen die Messetage raus.

In [ ]:
# TODO: Ergänzt den Merge so, dass alle Treffen bleiben und die Spalte _merge entsteht.
meetings = pairs.merge(candidates, on=["a", "b"])
marked = meetings.merge(fairs, how=___, on=["city_id", "checkin"], indicator=___)
outside_fairs = marked.query("_merge == 'left_only'")
print(f"{len(meetings):,} Treffen, davon {len(outside_fairs):,} außerhalb von Messen")

Zum Schluss zählen wir wie in Aufgabe 4.4 noch einmal, wenden Filter 1 erneut an und sortieren. Dieser Code ist
vorgegeben.

In [ ]:
result = outside_fairs.groupby(["a", "b"]).agg(meetings=("hotel_id", "size"), hotels=("hotel_id", "nunique"))
result = result[result["hotels"] >= 4].sort_values("meetings", ascending=False).reset_index()
result.head(15)

In [ ]:
check_result(result)

### Aufgabe 4.6: Wer steckt dahinter? · ⏱ ca. 10 Min.

Bisher sehen wir nur Nummern. Holen wir Namen und Firmen dazu. Weil jedes Paar **zwei** Gäste hat, verbinden
wir `guests` zweimal: einmal über `a`, einmal über `b`.

- `join(andere_tabelle, on="spalte")` sucht den Wert aus `spalte` im **Index** der anderen Tabelle.
- `add_suffix("_a")` hängt an alle Spaltennamen `_a` an, damit sich die Spalten der beiden Gäste nicht in die
  Quere kommen.

In [ ]:
# Mini-Beispiel
toy_result = pd.DataFrame({"a": [1, 2], "b": [3, 3], "meetings": [5, 2]})
toy_names = pd.DataFrame({"id": [1, 2, 3], "first_name": ["Ada", "Bo", "Cem"]}).set_index("id")
toy_result.join(toy_names.add_suffix("_a"), on="a").join(toy_names.add_suffix("_b"), on="b")

In [ ]:
# TODO: Holt die Namen für den zweiten Gast dazu.
names_by_id = guests.set_index("id")[["first_name", "last_name", "company"]]
named = result.join(names_by_id.add_suffix("_a"), on="a").join(names_by_id.add_suffix("_b"), on=___)
named.head(15)

🤔 Tauchen Personen mehrfach auf? Aus welchen Firmen kommen sie? Und welches Paar trifft sich auffällig oft?

### Aufgabe 4.7: Andere Blickwinkel · ⏱ ca. 20 Min.

Bisher hieß „zusammen“: selber Tag, selbes Hotel. Was, wenn jemand **nicht am selben Tag**, sondern immer
**einen Tag später** anreist? Das finden wir, indem wir ein Datum verschieben, bevor wir mergen.

`pd.Timedelta(days=1)` ist eine Zeitspanne von einem Tag. Man kann sie von Datumswerten abziehen oder dazuzählen.

In [ ]:
# Mini-Beispiel
toy_dates = pd.Series(pd.to_datetime(["2024-03-01", "2024-03-04"]))
pd.DataFrame({"original": toy_dates, "minus_1_tag": toy_dates - pd.Timedelta(days=1)})

Die Idee: Wir nennen die eine Seite `leader` (reist zuerst an) und die andere `follower`. Beim `follower`
ziehen wir einen Tag vom Anreisedatum ab. Reist der `follower` am 7. März an, steht dort danach der 6. März und
passt im Merge genau zum `leader`, der am 6. März angereist ist.

Hier gibt es kein `a < b`, denn die Reihenfolge ist jetzt wichtig: „B folgt A“ ist etwas anderes als „A folgt B“.
Wir entfernen nur Selbstpaare mit `leader != follower`.

In [ ]:
# TODO: Verschiebt die Anreise des follower um einen Tag nach vorne.
followers = guest_stays.rename(columns={"guest_id": "follower"})[["follower", "hotel_id", "checkin"]].copy()
followers["checkin"] = followers["checkin"] - pd.Timedelta(days=___)
leaders = guest_stays.rename(columns={"guest_id": "leader"})
lagged = leaders.merge(followers, on=["hotel_id", "checkin"]).query("leader != follower")
print(f"{len(lagged):,} Anreisen genau einen Tag nach einem anderen Gast")

Die Auswertung kennt ihr schon aus den Aufgaben 4.4 und 4.6. Sie ist vorgegeben.

In [ ]:
lag_stats = lagged.groupby(["leader", "follower"]).agg(trips=("hotel_id", "size"), hotels=("hotel_id", "nunique"))
lag_stats = lag_stats[lag_stats["hotels"] >= 4].sort_values("trips", ascending=False).reset_index()
lag_named = lag_stats.join(names_by_id.add_suffix("_leader"), on="leader")
lag_named = lag_named.join(names_by_id.add_suffix("_follower"), on="follower")
lag_named.head(5)

In [ ]:
check_lag_stats(lag_stats)

🤔 Was fällt euch beim obersten Paar auf? Welche Erklärungen gäbe es, harmlose und weniger harmlose?

**Zum Weiterprobieren (freiwillig):** Verändert die Definition noch einmal. Ideen:

- selbe **Stadt** statt selbes Hotel (`city_id` statt `hotel_id`)
- zwei Tage statt einem Tag Abstand
- mehr oder weniger als 4 verschiedene Hotels als Filter

In [ ]:
# Platz für eure Experimente

## Block 5: Darf man das?

In Block 4 habt ihr aus ganz gewöhnlichen Buchungsdaten Beziehungen zwischen Menschen herausgelesen. Technisch
war das nicht schwer: ein paar Merges, ein paar Filter. Die spannendere Frage lautet: **Darf man das überhaupt?**

### Aufgabe 5.1: Datenschutz und Fehlverdacht · ⏱ ca. 15 Min.

Alle Personen in diesen Daten sind frei erfunden. Bei echten Gästen sähe die Lage anders aus:

- **Zweckbindung (Art. 5 Abs. 1 lit. b DSGVO):** Buchungsdaten werden erhoben, um eine Übernachtung abzuwickeln.
  Dürfen wir daraus Beziehungen zwischen Gästen ableiten? Auch die **Datenminimierung** (Art. 5 Abs. 1 lit. c
  DSGVO) gehört in diese Diskussion: Es dürfen nur so viele Daten verarbeitet werden, wie für den Zweck nötig sind.
- **Sensible Rückschlüsse (Art. 9 DSGVO):** Aus harmlosen Daten können Hinweise auf Gesundheit, Religion,
  politische Haltung, Gewerkschaftszugehörigkeit oder sexuelle Orientierung entstehen. Solche Daten sind
  besonders geschützt.
- **Pseudonymisierung ist keine Anonymisierung:** Unser Export enthält keine E-Mail-Adressen, und selbst eine
  reine `guest_id` oder ein `card_token` verknüpft weiterhin alle Aufenthalte einer Person. Pseudonymisierte
  Daten bleiben personenbezogene Daten (Art. 4 Nr. 5 und Erwägungsgrund 26 DSGVO).
- **Fehlverdacht:** Wer sehr viele Paare prüft, findet auch dann Auffälligkeiten, wenn gar nichts dahintersteckt.

Wie viele Paare sind das eigentlich? Bei `n` Gästen kann jeder Gast mit `n - 1` anderen ein Paar bilden. Dabei
zählt man jedes Paar doppelt (A mit B und B mit A), also teilt man durch 2:

$$\text{Paare} = \frac{n \cdot (n - 1)}{2}$$

Ihr kennt das schon vom Kartell: 4 Personen ergeben 4 · 3 / 2 = 6 Paare.

In [ ]:
# TODO: Ergänzt die Formel für die Anzahl möglicher Paare.
guest_count = guests["id"].nunique()
possible_pairs = guest_count * (guest_count - 1) // ___
print(f"{guest_count:,} Gäste ergeben {possible_pairs:,} mögliche Paare.")
print(f"Schon bei einem Treffer auf eine Million Paare wären das {possible_pairs // 1_000_000:,} Verdachtsfälle.")

### Diskussion

Besprecht in kleinen Gruppen und notiert eure Antworten:

1. Wer in einem Hotelkonzern dürfte eine solche Auswertung starten, und wer müsste sie genehmigen?
2. Was würde passieren, wenn ein Treffer falsch ist und die betroffene Person davon erfährt?
3. Welche Spalten würdet ihr vor einem Export entfernen oder vergröbern, und welche Auswertungen gingen dann
   nicht mehr?
4. Wo verläuft für euch die Grenze zwischen Preisoptimierung und Überwachung?

## Geschafft!

Ihr habt ein ganzes Analyse-Projekt mit echten Fragen durchgespielt. Ein kurzer Rückblick:

- **pandas-Werkzeuge:** Tabellen filtern, neue Spalten berechnen, mit `groupby` und `agg` zusammenfassen, mit
  `transform` Gruppenwerte in jede Zeile schreiben, mit `merge` Tabellen verbinden (auch mit sich selbst und als
  Anti-Join), mit `pd.Timedelta` Datumswerte verschieben und Ergebnisse als Plot zeigen.
- **SQL und pandas:** Vieles geht in beiden Welten. SQL rechnet direkt in der Datenbank, pandas ist bequem zum
  Ausprobieren und Visualisieren.
- **Vorsicht bei Zusammenhängen:** Ein Muster ist noch keine Ursache. Die Osterfalle, die Preisfalle und die
  Berater in Block 4 zeigen: Erst wenn ihr harmlose Erklärungen ausgeschlossen habt, wird ein Befund belastbar.
- **Datenschutz:** Was technisch einfach geht, ist nicht automatisch erlaubt. Zweck, Datenminimierung und das
  Risiko eines Fehlverdachts gehören zu jeder Analyse dazu.